## **Project Information**
## **Authors: Shrey Patel (sp2675), Abhishek Jani (aj1121), Mustafa Adil (ma2398)**
- Note:
This code was developed and tested using Google Colab.For best results and to ensure all dependencies are handled correctly,it is recommended to upload the corresponding .ipynb file to Google Colab and run the cells sequentially.



In [2]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
# Copy model and files back from Drive
shutil.copytree('/content/drive/MyDrive/CS671_ADE_complete', '/content/', dirs_exist_ok=True)
print("Restored:", os.listdir('/content/'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Restored: ['.config', 'trained_model', 'brand_nonflip_examples_ade_only.json', 'extracted_drug_names.json', 'classification_data_with_brands.parquet', 'drive', 'dataset_created_used.txt', 'final_results', 'ade_project_outputs.zip', 'brand_flip_examples_ade_only.json', 'masking_non_flip_examples_ade_only.json', 'masking_flip_examples_ade_only.json', 'brand_nonflip_examples.json', 'brand_flip_examples.json', 'output', 'results_quick', 'full_drug_brand_mapping.json', 'sample_data']


# **Setup and Installs**

In [3]:
# !pip install --upgrade numpy==1.26.4 # Specific numpy version if required by dependencies
# !pip install --upgrade thinc==8.3.5 # Specific thinc version if required by dependencies
!pip install --upgrade pandas scikit-learn transformers datasets textattack requests tqdm

import pandas as pd
import numpy as np
import torch
import re
import json
import os
import time
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm # for progress bars

# Import components from libraries
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    AutoConfig
)
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from textattack.models.wrappers import HuggingFaceModelWrapper # For TextAttack integration
tqdm.pandas()
print("libraries imported")

gpu_available = torch.cuda.is_available()
if gpu_available:
  print("gpu is available")
  DEVICE = torch.device("cuda")
else:
    print("No gpu, using cpu")
    DEVICE = torch.device("cpu")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 90.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 73.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 118.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 445.7/445.7 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

textattack: Updating TextAttack package dependencies.
textattack: Downloading NLTK required packages.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package omw to /root/nltk_data...
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Unzipping taggers/universal_tagset.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape s

libraries imported
gpu is available


# **Configuration**

```
Note: Seeding for reproducibility is often done here, but currently commented out.

Uncomment these lines if you need strict reproducibility across runs.

np.random.seed(CONFIG["random_seed"])
torch.manual_seed(CONFIG["random_seed"])
if torch.cuda.is_available():
  torch.cuda.manual_seed_all(CONFIG["random_seed"])
```

In [4]:
CONFIG = {
    #dataset
    "classification_data_path": "hf://datasets/ade-benchmark-corpus/ade_corpus_v2/Ade_corpus_v2_classification/train-00000-of-00001.parquet",
    "relation_data_path": "hf://datasets/ade-benchmark-corpus/ade_corpus_v2/Ade_corpus_v2_drug_ade_relation/train-00000-of-00001.parquet",
    "extracted_drugs_file": "extracted_drug_names.json",
    "brand_mapping_file": "full_drug_brand_mapping.json",
    "modified_data_file": "classification_data_with_brands.parquet",
    #model
    "base_model_name": "dmis-lab/biobert-base-cased-v1.1",
    "finetuned_model_dir": "./trained_model",
    "tokenizer_max_length": 128,
    "num_labels": 2,

    "training_output_dir": "./results_quick",
    "num_train_epochs": 10,
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 16,
    "save_strategy_finetune": "epoch",
    "logging_steps": 200,
    "report_to": "none",
    "dataloader_num_workers": 2,
    "fp16_training": torch.cuda.is_available(),

    "ade_label": 1,
    "test_split_size": 0.2,
    "random_seed": 42,
    #api
    "api_delay_seconds": 0.2,
    "api_timeout_seconds": 10,
    #ouput files
    "masking_flip_ade_file": "masking_flip_examples_ade_only.json",
    "masking_nonflip_ade_file": "masking_non_flip_examples_ade_only.json",
    "masking_flip_non_ade_file": "output/masking_non_ade_flips.json",
    "masking_nonflip_non_ade_file": "output/masking_non_ade_nonflips.json",
    "brand_flip_all_file": "brand_flip_examples.json",
    "brand_nonflip_all_file": "brand_nonflip_examples.json",
    "brand_flip_ade_file": "brand_flip_examples_ade_only.json",
    "brand_nonflip_ade_file": "brand_nonflip_examples_ade_only.json",
}

# np.random.seed(CONFIG["random_seed"])
# torch.manual_seed(CONFIG["random_seed"])
# if torch.cuda.is_available():
#     torch.cuda.manual_seed_all(CONFIG["random_seed"])

# **Utility Functions**

In [5]:
def predict_label(text, model_wrapper):
    if not model_wrapper:
         raise ValueError("Model wrapper is not available.")
    try:
        # Ensuring that the text is a non-empty string
        if not isinstance(text, str): text = str(text)
        if not text or text.strip() == "":
            return 0 # default  for empty or invalid input
        outputs = model_wrapper([text])
        logits = outputs[0]
        if isinstance(logits, torch.Tensor):
            logits = logits.cpu().detach().numpy()
        # for handling NaN/inf in logits if model outputs them
        if not np.all(np.isfinite(logits)):
             print(f"Warning: Non-finite logits encountered for text: '{text[:50]}...'. Returning default (0).")
             return 0
        return np.argmax(logits)
    except Exception as e:
        print(f"Error during prediction for text: '{text[:50]}...': {e}")
        return 0 # on error return default

In [6]:
def save_json(data, filename):
    try:
        output_dir = os.path.dirname(filename)
        if output_dir and not os.path.exists(output_dir):
            os.makedirs(output_dir)
        with open(filename, "w", encoding='utf-8') as f:
            json.dump(data, f, indent=4, ensure_ascii=False)
    except Exception as e:
        print(f"Had an error saving data to {filename}: {e}")

In [7]:
def load_json(filename):
    if not os.path.exists(filename):
        print(f"file not found {filename}")
        return None
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            data = json.load(f)
        return data
    except json.JSONDecodeError as e:
        print(f"error decoding {filename}: {e}")
        return None
    except Exception as e:
        print(f"erroe reading {filename}: {e}")
        return None

In [8]:
def check_and_create_dir(directory_path):# func checks if a dir exists, if it does not exist then creates it.
    if directory_path and not os.path.exists(directory_path):
        print(f"Creating directory {directory_path}")
        os.makedirs(directory_path)

# **Data Loading and Preprocessing Functions**

In [9]:
def load_parquet_data(path):
    try:
        df = pd.read_parquet(path)
        print(f"loaded {len(df)} rows from {path}")
        return df
    except Exception as e:
        print(f"Error loading parquet file from {path}: {e}")
        return None

In [10]:
def extract_drug_names(df_relation, drug_col='drug'):
    if df_relation is None or drug_col not in df_relation.columns:
        print(f"Error in drug data '{drug_col}'.")
        return []
    unique_drugs = set()
    for drugs_entry in df_relation[drug_col].dropna():
        for drug in str(drugs_entry).split(','):
            drug_clean = drug.strip().lower()
            if drug_clean and len(drug_clean) > 1: # Adding basic validation
                unique_drugs.add(drug_clean)
    sorted_drugs = sorted(list(unique_drugs))
    print(f"there are {len(sorted_drugs)} unique drug names")
    return sorted_drugs

In [11]:
def tokenize_function_hf(examples, tokenizer, max_length, text_col="text"):
    return tokenizer(
        examples[text_col],
        truncation=True,
        padding="max_length",
        max_length=max_length
    )

In [12]:
def prepare_hf_dataset(df, tokenizer, config, text_col="text", label_col="label"):
    if df is None or text_col not in df.columns or label_col not in df.columns:
         print("Error in dataset creation")
         return None, None # Return None for both training and test

    dataset = Dataset.from_pandas(df[[text_col, label_col]])

    print("Splitting dataset")
    split = dataset.train_test_split(test_size=config["test_split_size"], seed=config["random_seed"])
    train_dataset = split["train"]
    test_dataset = split["test"]

    #tokenizing
    train_dataset = train_dataset.map(
        lambda examples: tokenize_function_hf(examples, tokenizer, config["tokenizer_max_length"], text_col),
        batched=True
    )
    test_dataset = test_dataset.map(
        lambda examples: tokenize_function_hf(examples, tokenizer, config["tokenizer_max_length"], text_col),
        batched=True
    )

    try:
        train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", label_col])
        test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", label_col])
        train_dataset = train_dataset.rename_column(label_col, "labels")
        test_dataset = test_dataset.rename_column(label_col, "labels")
    except Exception as e:
         print(f"error{e}")
         return None, None

    return train_dataset, test_dataset

# **Model Functions**

In [13]:
def load_tokenizer_and_model(model_name_or_path, num_labels):
    try:
        print(f"loading tokenizer from {model_name_or_path}")
        tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)

        print(f"loading model from {model_name_or_path}")
        model_config = AutoConfig.from_pretrained(
            model_name_or_path,
            num_labels=num_labels
        )
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name_or_path,
            config=model_config
        )
        model.to(DEVICE)
        print("loaded successfully")
        return tokenizer, model
    except Exception as e:
        print(f"Error loading{model_name_or_path}: {e}")
        return None, None

In [14]:
def create_model_wrapper(model, tokenizer):
    if model is None or tokenizer is None:
        print("Error, no model or tokenizer")
        return None
    try:
        model_wrapper = HuggingFaceModelWrapper(model, tokenizer)
        print("Model is wrapped successfully.")
        return model_wrapper
    except Exception as e:
        print(f"Error in wrapping model: {e}")
        return None

In [15]:
def compute_metrics_hf(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro')
    precision = precision_score(labels, predictions, average='macro', zero_division=0)
    recall = recall_score(labels, predictions, average='macro', zero_division=0)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

In [16]:
def train_model(model, tokenizer, train_dataset, eval_dataset, config):
    if model is None or tokenizer is None or train_dataset is None or eval_dataset is None:
        print("Error in training")
        return None

    training_args = TrainingArguments(
        output_dir=config["training_output_dir"],
        num_train_epochs=config["num_train_epochs"],
        per_device_train_batch_size=config["per_device_train_batch_size"],
        per_device_eval_batch_size=config["per_device_eval_batch_size"],
        save_strategy=config["save_strategy_finetune"],
        logging_steps=config["logging_steps"],
        report_to=config["report_to"],
        fp16=config["fp16_training"],
        dataloader_num_workers=config["dataloader_num_workers"],
        eval_strategy="epoch" if config["save_strategy_finetune"] != "no" else "no",
        logging_strategy="epoch",
        load_best_model_at_end=True if config["save_strategy_finetune"] != "no" else False,
        metric_for_best_model="f1" if config["save_strategy_finetune"] != "no" else None,
        seed=config["random_seed"]
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics_hf
    )

    print("Starting model finetuning")
    train_result = trainer.train()
    print("finished model finetuning")

    print("test set evaluation")
    eval_results = trainer.evaluate()
    print("Evaluation metrics:", eval_results)

    save_directory = config["finetuned_model_dir"]
    check_and_create_dir(save_directory)
    print(f"Saving model and tokenizer to: {save_directory}")
    trainer.save_model(save_directory)

    print("Model training and saving complete.")
    return model

# **Analysis Functions**

In [17]:
def mask_drug_names(text, drug_names_list, mask_token='[MASK]'):
    if not isinstance(text, str) or not drug_names_list:
        return text, False
    masked_text = text
    found = False
    sorted_drug_names = sorted(drug_names_list, key=len, reverse=True)

    for drug in sorted_drug_names:
        if not drug: continue
        try:
            #using \b for word boundaries
            pattern = re.compile(r'\b' + re.escape(drug) + r'\b', flags=re.IGNORECASE)
            temp_masked_text = pattern.sub(mask_token, masked_text)
            if temp_masked_text != masked_text:
                found = True
                masked_text = temp_masked_text
        except re.error as e:
            continue
    return masked_text, found

In [18]:
def prepare_brand_replacement(mapping_dict):
    if not mapping_dict:
        return {}, [], {}
    # Ensuring the keys are lowercase for consistent matching
    lower_mapping_dict = {k.lower(): v for k, v in mapping_dict.items()}
    compiled_patterns = {
        generic_lower: re.compile(r'\b' + re.escape(generic_lower) + r'\b', flags=re.IGNORECASE)
        for generic_lower in lower_mapping_dict.keys()
    }
    sorted_generic_keys = sorted(lower_mapping_dict.keys(), key=len, reverse=True)
    return lower_mapping_dict, compiled_patterns, sorted_generic_keys

In [19]:
def replace_generic_with_brand(text, lower_mapping_dict, compiled_patterns, sorted_keys):
    if not isinstance(text, str) or not lower_mapping_dict:
        return text, False

    current_text = text
    replaced = False
    for generic_lower in sorted_keys:
        pattern = compiled_patterns.get(generic_lower)
        brand_name = lower_mapping_dict.get(generic_lower)
        if pattern and brand_name:
            new_text = pattern.sub(brand_name, current_text)
            if new_text != current_text:
                replaced = True
                current_text = new_text
    return current_text, replaced

In [20]:
def calculate_flip_rate(df, text_col1, text_col2, model_wrapper, description="Processing"):
    if df is None or df.empty or model_wrapper is None:
        print(f"'{description}': invalid inputs.")
        return 0.0, [], []

    flip_count = 0
    total_processed = 0
    flip_examples = []
    non_flip_examples = []

    print(f"\nCalculating flip rate for: {description}")
    for index, row in tqdm(df.iterrows(), total=len(df), desc=description):
        text1 = row[text_col1]
        text2 = row[text_col2]

        #Basic validation
        if not isinstance(text1, str) or not isinstance(text2, str) or not text1 or not text2:
            continue

        try:
            pred1 = predict_label(text1, model_wrapper)
            pred2 = predict_label(text2, model_wrapper)
            total_processed += 1

            example_data = {
                'index': index,
                text_col1: text1,
                text_col2: text2,
                'prediction1': int(pred1),
                'prediction2': int(pred2)
            }
            if 'label' in row: example_data['original_label'] = int(row['label'])
            if pred1 != pred2:
                flip_count += 1
                flip_examples.append(example_data)
            else:
                non_flip_examples.append(example_data)
        except Exception as e:
            print(f"error in flip calculation for index {index} in {description}: {e}")

    flip_rate = flip_count / total_processed if total_processed > 0 else 0.0
    print(f"Results ({description}) ---")
    print(f"Total sentences processed: {total_processed}")
    print(f"sentences where prediction FLIPPED: {flip_count}")
    print(f"sentences where prediction DID NOT FLIP: {len(non_flip_examples)}")
    print(f"Flip Rate: {flip_rate:.4f}")

    return flip_rate, flip_examples, non_flip_examples

# **API Functions (RxNorm)**

In [21]:
def get_rxcui(generic_name, config):
    base_url = "https://rxnav.nlm.nih.gov/REST/rxcui.json"
    params = {'name': generic_name, 'search': 2} #Approx match
    try:
        response = requests.get(base_url, params=params, timeout=config["api_timeout_seconds"])
        response.raise_for_status()
        data = response.json()
        if 'idGroup' in data and 'rxnormId' in data['idGroup'] and data['idGroup']['rxnormId']:
            return data['idGroup']['rxnormId'][0] #Return first RxCUI
        return None
    except requests.exceptions.Timeout:
        print(f"Timeout fetching RxCUI for '{generic_name}'")
        return None
    except requests.exceptions.RequestException as e:
        print(f"API request failed for RxCUI lookup '{generic_name}': {e}")
        return None
    except Exception as e:
        print(f"Unexpected error in get_rxcui for '{generic_name}': {e}")
        return None

In [22]:
def get_brand_names(rxcui, config):
    if not rxcui: return []
    base_url = f"https://rxnav.nlm.nih.gov/REST/rxcui/{rxcui}/related.json"
    params = {'tty': 'BN'}
    brands = []
    try:
        response = requests.get(base_url, params=params, timeout=config["api_timeout_seconds"])
        response.raise_for_status()
        data = response.json()
        if ('relatedGroup' in data and 'conceptGroup' in data['relatedGroup']):
            for group in data['relatedGroup']['conceptGroup']:
                if group.get('tty') == 'BN' and 'conceptProperties' in group:
                    for prop in group['conceptProperties']:
                        if 'name' in prop: brands.append(prop['name'])
    except requests.exceptions.HTTPError as http_err:
        if response.status_code != 404:
             print(f"HTTP error fetching brands for RxCUI {rxcui}: {http_err}")
    except requests.exceptions.Timeout:
        print(f"Timeout")
    except requests.exceptions.RequestException as e:
        print(f"API request failed: {e}")
    except Exception as e:
        print(f"error")
    return brands

In [24]:
def fetch_brand_mappings(generic_names_list, config):
    if not generic_names_list: return []
    all_mappings = []
    processed_count = 0
    found_count = 0
    print(f"API lookups for {len(generic_names_list)} names")

    for name in tqdm(generic_names_list, desc="Fetching Brand Mappings"):
        processed_count += 1
        if not isinstance(name, str) or not name.strip() or len(name.strip()) < 2:
             continue #Skiping over invalid names

        generic_name_cleaned = name.strip()
        current_rxcui = get_rxcui(generic_name_cleaned, config)
        time.sleep(config["api_delay_seconds"]) #Delay

        if current_rxcui:
            brands = get_brand_names(current_rxcui, config)
            time.sleep(config["api_delay_seconds"]) #Delay

            if brands:
                selected_brand = brands[0].strip().lower()
                if selected_brand:
                    all_mappings.append({'generic_name': name, 'brand_name': selected_brand})
                    found_count += 1

    print(f"API lookups completed and found {found_count}/{len(generic_names_list)} names")
    return all_mappings

# **Workflow - Step 1: Fine-tuning**

In [ ]:
df_classification = load_parquet_data(CONFIG["classification_data_path"])
if df_classification is not None:
    tokenizer, model = load_tokenizer_and_model(CONFIG["base_model_name"], CONFIG["num_labels"])

    if tokenizer and model:
        train_ds, test_ds = prepare_hf_dataset(df_classification, tokenizer, CONFIG)

        if train_ds and test_ds:
            model = train_model(model, tokenizer, train_ds, test_ds, CONFIG)
            if model:
                print("Finetuning completed")
            else:
                print("Model training failed")
        else:
            print("Dataset preparation failed")
    else:
        print("Base model or tokenizer loading failed")
else:
    print("Classification data loading failed")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


loaded 23516 rows from hf://datasets/ade-benchmark-corpus/ade_corpus_v2/Ade_corpus_v2_classification/train-00000-of-00001.parquet
loading tokenizer from dmis-lab/biobert-base-cased-v1.1


config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

loading model from dmis-lab/biobert-base-cased-v1.1


...

/tmp/ipykernel_1788/573097909.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Starting model finetuning


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.198900,0.182339,0.947704,0.934749,0.944875,0.925929
2,0.107600,0.152852,0.957696,0.948349,0.948053,0.948647
3,0.061400,0.228858,0.957483,0.948261,0.946522,0.950047
4,0.036900,0.237956,0.958333,0.949751,0.944288,0.955733
5,0.019600,0.236543,0.960247,0.951781,0.948638,0.955083
6,0.010200,0.311362,0.959396,0.950728,0.947774,0.953823
7,0.009000,0.295646,0.960034,0.951280,0.950296,0.952279
8,0.003700,0.336478,0.961310,0.952519,0.954581,0.950517
9,0.000500,0.374671,0.962160,0.953831,0.953233,0.954433
10,0.000400,0.392514,0.960034,0.951450,0.948950,0.954049


finished model finetuning
test set evaluation


Evaluation metrics: {'eval_loss': 0.3746706247329712, 'eval_accuracy': 0.9621598639455783, 'eval_f1': 0.9538305076222406, 'eval_precision': 0.9532332075152514, 'eval_recall': 0.9544331809448088, 'eval_runtime': 4.5899, 'eval_samples_per_second': 1024.856, 'eval_steps_per_second': 64.053, 'epoch': 10.0}
Creating directory ./trained_model
Saving model and tokenizer to: ./trained_model
Model training and saving complete.
Finetuning completed


# **Workflow - Step 2: Drug Name Extraction and Saving**

In [ ]:
df_relation = load_parquet_data(CONFIG["relation_data_path"])
extracted_drugs = []
if df_relation is not None:
    extracted_drugs = extract_drug_names(df_relation)
    if extracted_drugs:
        save_json(extracted_drugs, CONFIG["extracted_drugs_file"])
        print(f"Saved {len(extracted_drugs)} drug names to {CONFIG['extracted_drugs_file']}")
    else:
        print("No drug names were extracted")
else:
    print("Relation data loading failed")

loaded 6821 rows from hf://datasets/ade-benchmark-corpus/ade_corpus_v2/Ade_corpus_v2_drug_ade_relation/train-00000-of-00001.parquet
there are 1049 unique drug names
Saved 1049 drug names to extracted_drug_names.json


# **Workflow - Step 3: Drug Masking Analysis (ADE Sentences)**

In [ ]:
if 'df_classification' not in locals() or df_classification is None:
    print("reloading classification data for masking analysis")
    df_classification = load_parquet_data(CONFIG["classification_data_path"])

if 'extracted_drugs' not in locals() or not extracted_drugs:
     print(f"Loading extracted drug names from {CONFIG['extracted_drugs_file']}")
     extracted_drugs = load_json(CONFIG["extracted_drugs_file"])

if 'model' not in locals() or 'tokenizer' not in locals() or model is None or tokenizer is None:
    print(f"loading finetuned model from {CONFIG['finetuned_model_dir']}...")
    tokenizer, model = load_tokenizer_and_model(CONFIG["finetuned_model_dir"], CONFIG["num_labels"])

#Creating model wrapper or reusing
if 'model_wrapper' not in locals() or model_wrapper is None:
    model_wrapper = create_model_wrapper(model, tokenizer)


if df_classification is not None and extracted_drugs and model_wrapper and tokenizer:
    # Filter for ADE sentences
    ade_df = df_classification[df_classification['label'] == CONFIG['ade_label']].copy()
    print(f"Processing {len(ade_df)} ADE sentences for masking analysis")

    if not ade_df.empty:
        # Apply masking
        mask_results = ade_df['text'].progress_apply(
            lambda x: mask_drug_names(x, extracted_drugs, tokenizer.mask_token)
        )
        ade_df['masked_text'] = mask_results.apply(lambda x: x[0])
        ade_df['drug_found_flag'] = mask_results.apply(lambda x: x[1])

        # Filter df to rows where drug was actually found and masked
        ade_df_masked = ade_df[ade_df['drug_found_flag'] == True].copy()
        print(f"Found drugs and applied masking to {len(ade_df_masked)} ADE sentences.")

        if not ade_df_masked.empty:
             # Calculate flip rate
             _, flip_examples, non_flip_examples = calculate_flip_rate(
                  ade_df_masked, 'text', 'masked_text', model_wrapper, "Masking (ADE Only)"
             )
             # Save results
             save_json(flip_examples, CONFIG["masking_flip_ade_file"])
             save_json(non_flip_examples, CONFIG["masking_nonflip_ade_file"])
             print(f"Saved masking analysis examples for ADE sentences.")
        else:
             print("No ADE sentences had detectable drug names from the list. Skipping flip rate calculation.")
    else:
        print("No ADE sentences found in the dataset.")
else:
    print("Skipping masking analysis due to missing data, drug names, or model.")

Model is wrapped successfully.
Processing 6821 ADE sentences for masking analysis


  0%|          | 0/6821 [00:00<?, ?it/s]

Found drugs and applied masking to 6821 ADE sentences.

Calculating flip rate for: Masking (ADE Only)


Masking (ADE Only):   0%|          | 0/6821 [00:00<?, ?it/s]

Results (Masking (ADE Only)) ---
Total sentences processed: 6821
sentences where prediction FLIPPED: 5084
sentences where prediction DID NOT FLIP: 1737
Flip Rate: 0.7453
Saved masking analysis examples for ADE sentences.


# **Workflow - Step 4: Drug Masking Analysis (Non ADE Sentences)**

In [ ]:
if 'df_classification' not in locals() or df_classification is None:
    print("Reloading classification data for masking analysis...")
    df_classification = load_parquet_data(CONFIG["classification_data_path"])

if 'extracted_drugs' not in locals() or not extracted_drugs:
     print(f"Loading extracted drug names from {CONFIG['extracted_drugs_file']}")
     extracted_drugs = load_json(CONFIG["extracted_drugs_file"])

if 'model' not in locals() or 'tokenizer' not in locals() or model is None or tokenizer is None:
    print(f"Loading finetuned model from {CONFIG['finetuned_model_dir']}")
    tokenizer, model = load_tokenizer_and_model(CONFIG["finetuned_model_dir"], CONFIG["num_labels"])

if 'model_wrapper' not in locals() or model_wrapper is None:
    model_wrapper = create_model_wrapper(model, tokenizer)

if "masking_flip_non_ade_file" not in CONFIG or "masking_nonflip_non_ade_file" not in CONFIG:
    raise KeyError("Config Error")

if df_classification is not None and extracted_drugs and model_wrapper and tokenizer:
    # Filter for Non-ADE sentences
    non_ade_df = df_classification[df_classification['label'] != CONFIG['ade_label']].copy()
    print(f"Processing {len(non_ade_df)}")

    if not non_ade_df.empty:
        mask_results = non_ade_df['text'].progress_apply(
            lambda x: mask_drug_names(x, extracted_drugs, tokenizer.mask_token)
        )
        non_ade_df['masked_text'] = mask_results.apply(lambda x: x[0])
        non_ade_df['drug_found_flag'] = mask_results.apply(lambda x: x[1])
        non_ade_df_masked = non_ade_df[non_ade_df['drug_found_flag'] == True].copy()
        print(f"Found drugs and applied masking to {len(non_ade_df_masked)} Non-ADE sentences")

        if not non_ade_df_masked.empty:
             # Calculate flip rate

             _, flip_examples, non_flip_examples = calculate_flip_rate(
                  non_ade_df_masked, 'text', 'masked_text', model_wrapper, "Masking (Non-ADE Only)"
             )
             # Save results
             save_json(flip_examples, CONFIG["masking_flip_non_ade_file"])
             save_json(non_flip_examples, CONFIG["masking_nonflip_non_ade_file"])
             print(f"Saved masking analysis examples for Non-ADE sentences.")
        else:
             print("No drug names")
    else:
        print("No Non-ADE sentences found in the dataset")
else:
    print("error in masking analysis due to missing data")

Processing 16695


  0%|          | 0/16695 [00:00<?, ?it/s]

Found drugs and applied masking to 5093 Non-ADE sentences

Calculating flip rate for: Masking (Non-ADE Only)


Masking (Non-ADE Only):   0%|          | 0/5093 [00:00<?, ?it/s]

Results (Masking (Non-ADE Only)) ---
Total sentences processed: 5093
sentences where prediction FLIPPED: 84
sentences where prediction DID NOT FLIP: 5009
Flip Rate: 0.0165
Saved masking analysis examples for Non-ADE sentences.


# **Workflow - Step 5: Fetch and Save Brand Name Mappings**

In [ ]:
if 'extracted_drugs' not in locals() or not extracted_drugs:
     print(f"Loading extracted drug names from {CONFIG['extracted_drugs_file']}")
     extracted_drugs = load_json(CONFIG["extracted_drugs_file"])

brand_mappings_list = []
if extracted_drugs:
    brand_mappings_list = fetch_brand_mappings(extracted_drugs, CONFIG)
    if brand_mappings_list:
        save_json(brand_mappings_list, CONFIG["brand_mapping_file"])
        print(f"Saved {len(brand_mappings_list)} brand mappings to {CONFIG['brand_mapping_file']}")
    else:
        print("No brand mappings were etched")
else:
    print("extracted drug list is missing")

API lookups for 1049 names


Fetching Brand Mappings:   0%|          | 0/1049 [00:00<?, ?it/s]

API lookups completed and found 540/1049 names
Saved 540 brand mappings to full_drug_brand_mapping.json


# **Workflow - Step 6: Apply Brand Name Replacement**

In [ ]:
if 'df_classification' not in locals() or df_classification is None or 'text_brand_replaced' in df_classification.columns:
    print(f"Loading classification data from {CONFIG['classification_data_path']}")
    df_classification = load_parquet_data(CONFIG["classification_data_path"])

if 'brand_mappings_list' not in locals() or not brand_mappings_list:
     print(f"loading brand mappings from {CONFIG['brand_mapping_file']}")
     brand_mappings_list = load_json(CONFIG["brand_mapping_file"])

if df_classification is not None and brand_mappings_list:
    brand_mapping_dict = {item['generic_name'].lower(): item['brand_name']
                          for item in brand_mappings_list if 'generic_name' in item and 'brand_name' in item}

    if brand_mapping_dict:
        #Preparing for replacement
        lower_map_dict, compiled_patterns, sorted_keys = prepare_brand_replacement(brand_mapping_dict)

        print("Applying generic-to-brand replacement")
        #Applying replacement function
        replacement_results = df_classification['text'].progress_apply(
            lambda x: replace_generic_with_brand(x, lower_map_dict, compiled_patterns, sorted_keys)
        )
        df_classification['text_brand_replaced'] = replacement_results.apply(lambda x: x[0])
        df_classification['brand_replaced_flag'] = replacement_results.apply(lambda x: x[1])
        replaced_count = df_classification['brand_replaced_flag'].sum()
        print(f"{replaced_count} sentences had replacements")
        try:
            check_and_create_dir(os.path.dirname(CONFIG["modified_data_file"])) # Ensure dir exists
            df_classification.to_parquet(CONFIG["modified_data_file"], index=False)
            print(f"saved modified datato {CONFIG['modified_data_file']}")
        except Exception as e:
            print(f"error saving modified data: {e}")
    else:
        print("brand drug names dictionary is empty")
else:
    print("missing data")

Applying generic-to-brand replacement


  0%|          | 0/23516 [00:00<?, ?it/s]

8412 sentences had replacements
saved modified datato classification_data_with_brands.parquet


# **Workflow - Step 7: Brand Replacement Flip Rate (All Sentences)**


In [ ]:
if 'df_classification' not in locals() or 'brand_replaced_flag' not in df_classification.columns:
     print(f"Loading modified data from {CONFIG['modified_data_file']}")
     # Check if file exists before loading
     if os.path.exists(CONFIG['modified_data_file']):
         df_classification = load_parquet_data(CONFIG["modified_data_file"])
     else:
         print(f"data not found {CONFIG['modified_data_file']}")
         df_classification = None

if 'model_wrapper' not in locals() or model_wrapper is None:
    print(f"Reloading fine-tuned model from {CONFIG['finetuned_model_dir']}...")
    tokenizer, model = load_tokenizer_and_model(CONFIG["finetuned_model_dir"], CONFIG["num_labels"])
    model_wrapper = create_model_wrapper(model, tokenizer)


if df_classification is not None and 'brand_replaced_flag' in df_classification.columns and model_wrapper:
    # Filter for rows where replacement actually happened
    df_replaced_all = df_classification[df_classification['brand_replaced_flag'] == True].copy()
    print(f"Analyzing {len(df_replaced_all)} sentences where brand replacement occurred.")

    if not df_replaced_all.empty:
         # Calculate flip rate
         _, flip_examples, non_flip_examples = calculate_flip_rate(
              df_replaced_all, 'text', 'text_brand_replaced', model_wrapper, "Brand Replace (All)"
         )
         # Save results
         save_json(flip_examples, CONFIG["brand_flip_all_file"])
         save_json(non_flip_examples, CONFIG["brand_nonflip_all_file"])
         print(f"Saved brand replacement flip analysis")
    else:
         print("No brand replacements")
else:
    print("missing data")

Analyzing 8412 sentences where brand replacement occurred.

Calculating flip rate for: Brand Replace (All)


Brand Replace (All):   0%|          | 0/8412 [00:00<?, ?it/s]

Results (Brand Replace (All)) ---
Total sentences processed: 8412
sentences where prediction FLIPPED: 748
sentences where prediction DID NOT FLIP: 7664
Flip Rate: 0.0889
Saved brand replacement flip analysis


# **Workflow - Step 8: Brand Replacement Flip Rate (ADE Sentences Only)**

In [ ]:
required_cols = ['label', 'brand_replaced_flag', 'text', 'text_brand_replaced']
if 'df_classification' not in locals() or not all(col in df_classification.columns for col in required_cols):
     print(f"Loading modified data from {CONFIG['modified_data_file']}")
     if os.path.exists(CONFIG['modified_data_file']):
        df_classification = load_parquet_data(CONFIG["modified_data_file"])
     else:
         print(f"data not found {CONFIG['modified_data_file']}")
         df_classification = None

if 'model_wrapper' not in locals() or model_wrapper is None:
    print(f"reloading finetuned model from {CONFIG['finetuned_model_dir']}")
    tokenizer, model = load_tokenizer_and_model(CONFIG["finetuned_model_dir"], CONFIG["num_labels"])
    model_wrapper = create_model_wrapper(model, tokenizer)

if df_classification is not None and all(col in df_classification.columns for col in required_cols) and model_wrapper:
    df_replaced_ade = df_classification[
        (df_classification['brand_replaced_flag'] == True) &
        (df_classification['label'] == CONFIG['ade_label'])
    ].copy()
    print(f"Analyzing {len(df_replaced_ade)} ADE sentences where brand replacement occurred.")

    if not df_replaced_ade.empty:
        #calculating flip rate
        _, flip_examples, non_flip_examples = calculate_flip_rate(
            df_replaced_ade, 'text', 'text_brand_replaced', model_wrapper, "Brand Replace (ADE Only)"
        )
        save_json(flip_examples, CONFIG["brand_flip_ade_file"])
        save_json(non_flip_examples, CONFIG["brand_nonflip_ade_file"])
        print(f"Saved brand replacement flip analysis examples (ADE sentences only).")
    else:
        print("No ADE sentences found where brand replacement occurred. Skipping analysis.")
else:
    if df_classification is None:
         print("Skipping brand replacement flip analysis (ADE Only) because DataFrame is missing.")
    elif not all(col in df_classification.columns for col in required_cols):
         print(f"Skipping brand replacement flip analysis (ADE Only) because DataFrame is missing required columns: {required_cols}")
    elif model_wrapper is None:
         print("Skipping brand replacement flip analysis (ADE Only) because Model Wrapper is missing.")
    else:
         print("Skipping brand replacement flip analysis (ADE Only) due to missing data or model.")

Analyzing 5142 ADE sentences where brand replacement occurred.

Calculating flip rate for: Brand Replace (ADE Only)


Brand Replace (ADE Only):   0%|          | 0/5142 [00:00<?, ?it/s]

Results (Brand Replace (ADE Only)) ---
Total sentences processed: 5142
sentences where prediction FLIPPED: 710
sentences where prediction DID NOT FLIP: 4432
Flip Rate: 0.1381
Saved brand replacement flip analysis examples (ADE sentences only).


In [ ]:
import hashlib

def predict_batch(texts, batch_size=64):
    """Fast batched inference returning labels + probabilities."""
    all_labels, all_probs = [], []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(
            batch, padding=True, truncation=True,
            max_length=128, return_tensors='pt'
        )
        encoded = {k: v.to(DEVICE) for k, v in encoded.items()}
        with torch.no_grad():
            outputs = model(**encoded)
            probs = torch.softmax(outputs.logits, dim=-1)
            labels = torch.argmax(probs, dim=-1)
        all_labels.extend(labels.cpu().numpy().tolist())
        all_probs.extend(probs[:, 1].cpu().numpy().tolist())
    return np.array(all_labels), np.array(all_probs)

def sentence_id(text):
    return hashlib.sha1(text.encode()).hexdigest()[:12]

print("Batched inference function ready")

Batched inference function ready


In [ ]:
import json

with open('/content/masking_flip_examples_ade_only.json') as f:
    family_a_flips = json.load(f)

with open('/content/masking_non_flip_examples_ade_only.json') as f:
    family_a_nonflips = json.load(f)

all_ade_sentences = family_a_flips + family_a_nonflips

print(f"Total ADE sentences: {len(all_ade_sentences)}")
print(f"Flipped: {len(family_a_flips)}")
print(f"Non-flipped: {len(family_a_nonflips)}")
print(f"Sample keys: {list(all_ade_sentences[0].keys())}")

Total ADE sentences: 6821
Flipped: 5084
Non-flipped: 1737
Sample keys: ['index', 'text', 'masked_text', 'prediction1', 'prediction2', 'original_label']


In [ ]:
from tqdm import tqdm
from datetime import datetime

print("[Family A] Running with confidence scores...")

original_texts_a = [r['text'] for r in all_ade_sentences]
masked_texts_a   = [r['masked_text'] for r in all_ade_sentences]

orig_labels_a, orig_probs_a = predict_batch(original_texts_a)
mask_labels_a, mask_probs_a = predict_batch(masked_texts_a)

family_a_confidence = []
for i, record in enumerate(all_ade_sentences):
    flipped = bool(orig_labels_a[i] != mask_labels_a[i])
    family_a_confidence.append({
        "sentence_id":      sentence_id(record['text']),
        "index":            record['index'],
        "original_text":    record['text'],
        "masked_text":      record['masked_text'],
        "original_pred":    int(orig_labels_a[i]),
        "masked_pred":      int(mask_labels_a[i]),
        "original_prob":    float(orig_probs_a[i]),
        "masked_prob":      float(mask_probs_a[i]),
        "confidence_delta": float(orig_probs_a[i] - mask_probs_a[i]),
        "flipped":          flipped
    })

n_flipped_a   = sum(1 for r in family_a_confidence if r['flipped'])
flip_rate_a   = n_flipped_a / len(family_a_confidence)
mean_delta_a  = float(np.mean([r['confidence_delta'] for r in family_a_confidence]))
subthresh_a   = sum(1 for r in family_a_confidence
                    if r['confidence_delta'] >= 0.20 and not r['flipped'])

os.makedirs('/content/results', exist_ok=True)
with open('/content/results/family_a_with_confidence.json', 'w') as f:
    json.dump({
        "experiment": "family_a_drug_masking_ade",
        "n_sentences": len(family_a_confidence),
        "n_flipped": n_flipped_a,
        "flip_rate": flip_rate_a,
        "mean_confidence_delta": mean_delta_a,
        "subthreshold_drops_gte_020": subthresh_a,
        "per_sentence": family_a_confidence,
        "metadata": {"seed": 42, "timestamp": datetime.now().isoformat()}
    }, f, indent=2)

print(f"Flip rate:              {flip_rate_a:.1%} ({n_flipped_a}/{len(family_a_confidence)})")
print(f"Mean confidence delta:  {mean_delta_a:.3f}")
print(f"Sub-threshold drops:    {subthresh_a}")
print("Saved: /content/results/family_a_with_confidence.json")

[Family A] Running with confidence scores...
Flip rate:              74.5% (5084/6821)
Mean confidence delta:  0.745
Sub-threshold drops:    62
Saved: /content/results/family_a_with_confidence.json


In [ ]:
import pandas as pd

print("[Family B] Loading brand-replaced data...")
df_brands = pd.read_parquet('/content/classification_data_with_brands.parquet')

df_b_ade = df_brands[
    (df_brands['label'] == 1) &
    (df_brands['brand_replaced_flag'] == True)
].copy()
print(f"Family B ADE sentences: {len(df_b_ade)}")

original_texts_b = df_b_ade['text'].tolist()
replaced_texts_b = df_b_ade['text_brand_replaced'].tolist()

orig_labels_b, orig_probs_b = predict_batch(original_texts_b)
repl_labels_b, repl_probs_b = predict_batch(replaced_texts_b)

family_b_confidence = []
for i, (_, row) in enumerate(df_b_ade.iterrows()):
    flipped = bool(orig_labels_b[i] != repl_labels_b[i])
    family_b_confidence.append({
        "sentence_id":      sentence_id(row['text']),
        "original_text":    row['text'],
        "replaced_text":    row['text_brand_replaced'],
        "original_pred":    int(orig_labels_b[i]),
        "replaced_pred":    int(repl_labels_b[i]),
        "original_prob":    float(orig_probs_b[i]),
        "replaced_prob":    float(repl_probs_b[i]),
        "confidence_delta": float(orig_probs_b[i] - repl_probs_b[i]),
        "flipped":          flipped
    })

n_flipped_b  = sum(1 for r in family_b_confidence if r['flipped'])
flip_rate_b  = n_flipped_b / len(family_b_confidence)
mean_delta_b = float(np.mean([r['confidence_delta'] for r in family_b_confidence]))
subthresh_b  = sum(1 for r in family_b_confidence
                   if r['confidence_delta'] >= 0.20 and not r['flipped'])

with open('/content/results/family_b_with_confidence.json', 'w') as f:
    json.dump({
        "experiment": "family_b_brand_substitution_ade",
        "n_sentences": len(family_b_confidence),
        "n_flipped": n_flipped_b,
        "flip_rate": flip_rate_b,
        "mean_confidence_delta": mean_delta_b,
        "subthreshold_drops_gte_020": subthresh_b,
        "per_sentence": family_b_confidence,
        "metadata": {"seed": 42, "timestamp": datetime.now().isoformat()}
    }, f, indent=2)

print(f"Flip rate:             {flip_rate_b:.1%} ({n_flipped_b}/{len(family_b_confidence)})")
print(f"Mean confidence delta: {mean_delta_b:.3f}")
print(f"Sub-threshold drops:   {subthresh_b}")
print("Saved: /content/results/family_b_with_confidence.json")

[Family B] Loading brand-replaced data...
Family B ADE sentences: 5142
Flip rate:             13.8% (710/5142)
Mean confidence delta: 0.137
Sub-threshold drops:   13
Saved: /content/results/family_b_with_confidence.json


In [ ]:
from datasets import load_dataset

print("[Family C] Loading relation dataset for symptom extraction...")
relation_ds = load_dataset(
    'ade_corpus_v2',
    'Ade_corpus_v2_drug_ade_relation',
    trust_remote_code=True
)
df_relation = relation_ds['train'].to_pandas()

print(f"Columns: {df_relation.columns.tolist()}")
print(f"Total rows: {len(df_relation)}")
print(f"Sample effects: {df_relation['effect'].value_counts().head(10).to_dict()}")

# Build symptom vocabulary
symptom_terms = set()
for effect in df_relation['effect']:
    if not effect or effect.strip().lower() == 'none':
        continue
    term = effect.strip().lower()
    if len(term) > 1:
        symptom_terms.add(term)

# Sort longest first to avoid substring shadowing (mirrors drug masking)
symptom_list = sorted(symptom_terms, key=len, reverse=True)

print(f"\nExtracted {len(symptom_list)} unique symptom terms")
print(f"Sample terms: {symptom_list[:10]}")

with open('/content/results/extracted_symptom_terms.json', 'w') as f:
    json.dump(symptom_list, f, indent=2)
print("Saved: /content/results/extracted_symptom_terms.json")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ade_corpus_v2' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ade_corpus_v2' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


[Family C] Loading relation dataset for symptom extraction...


README.md: 0.00B [00:00, ?B/s]

Ade_corpus_v2_drug_ade_relation/train-00(…):   0%|          | 0.00/491k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6821 [00:00<?, ? examples/s]

Columns: ['text', 'drug', 'effect', 'indexes']
Total rows: 6821
Sample effects: {'fever': 61, 'thrombocytopenia': 56, 'neurotoxicity': 54, 'rhabdomyolysis': 44, 'seizures': 42, 'hepatotoxicity': 41, 'hepatitis': 39, 'acute renal failure': 32, 'agranulocytosis': 31, 'pneumonitis': 25}

Extracted 2983 unique symptom terms
Sample terms: ['reduce the gastrointestinal absorption of orally administered levothyroxine sodium', 'epstein-barr virus (ebv)-associated polyclonal b cell lymphoproliferative disorder', 'discrepancies between hematologic, marrow morphologic, and cytogenetic responses', 'bullous lesions on the hands and soles that disseminated, evolving to necrosis', 'significant 32%-46% loss of tyrosine hydroxylase (th) immunoreactive neurons', 'antineutrophil cytoplasmic antibody-positive crescentic glomerulonephritis', 'hearing impairment in the left ear (with progression to the right ear)', 'epstein-barr virus-associated polymorphic lymphoproliferative disorder', 'relapse in the ext

In [ ]:
print("[Family C] Running symptom masking on 6821 ADE sentences...")

def mask_symptoms(text, symptom_list, mask_token='[MASK]'):
    masked = text
    found = False
    for term in symptom_list:
        pattern = re.compile(r'\b' + re.escape(term) + r'\b', re.IGNORECASE)
        if pattern.search(masked):
            masked = pattern.sub(mask_token, masked)
            found = True
    return masked, found

# Build masked versions
family_c_input = []
excluded = 0

for record in tqdm(all_ade_sentences, desc="Masking symptoms"):
    masked_text, found = mask_symptoms(record['text'], symptom_list)
    if not found:
        excluded += 1
        continue
    family_c_input.append({
        "index":         record['index'],
        "original_text": record['text'],
        "masked_text":   masked_text
    })

print(f"\nSentences with symptom match: {len(family_c_input)}")
print(f"Sentences excluded (no symptom found): {excluded}")

# Run inference
original_texts_c = [r['original_text'] for r in family_c_input]
masked_texts_c   = [r['masked_text']   for r in family_c_input]

orig_labels_c, orig_probs_c = predict_batch(original_texts_c)
mask_labels_c, mask_probs_c = predict_batch(masked_texts_c)

# Build results
family_c_confidence = []
for i, record in enumerate(family_c_input):
    flipped = bool(orig_labels_c[i] != mask_labels_c[i])
    family_c_confidence.append({
        "sentence_id":      sentence_id(record['original_text']),
        "index":            record['index'],
        "original_text":    record['original_text'],
        "masked_text":      record['masked_text'],
        "original_pred":    int(orig_labels_c[i]),
        "masked_pred":      int(mask_labels_c[i]),
        "original_prob":    float(orig_probs_c[i]),
        "masked_prob":      float(mask_probs_c[i]),
        "confidence_delta": float(orig_probs_c[i] - mask_probs_c[i]),
        "flipped":          flipped
    })

n_flipped_c  = sum(1 for r in family_c_confidence if r['flipped'])
flip_rate_c  = n_flipped_c / len(family_c_confidence)
mean_delta_c = float(np.mean([r['confidence_delta'] for r in family_c_confidence]))
subthresh_c  = sum(1 for r in family_c_confidence
                   if r['confidence_delta'] >= 0.20 and not r['flipped'])

# THE HEADLINE METRIC
sensitivity_ratio = flip_rate_a / flip_rate_c if flip_rate_c > 0 else float('inf')

with open('/content/results/family_c_with_confidence.json', 'w') as f:
    json.dump({
        "experiment": "family_c_symptom_masking",
        "n_sentences_total_ade": len(all_ade_sentences),
        "n_sentences_with_symptom": len(family_c_confidence),
        "n_excluded_no_symptom": excluded,
        "n_flipped": n_flipped_c,
        "flip_rate": flip_rate_c,
        "mean_confidence_delta": mean_delta_c,
        "subthreshold_drops_gte_020": subthresh_c,
        "sensitivity_ratio": sensitivity_ratio,
        "family_a_flip_rate_for_ratio": flip_rate_a,
        "per_sentence": family_c_confidence,
        "metadata": {
            "symptom_vocab_size": len(symptom_list),
            "seed": 42,
            "timestamp": datetime.now().isoformat()
        }
    }, f, indent=2)

print(f"\nFlip rate:             {flip_rate_c:.1%} ({n_flipped_c}/{len(family_c_confidence)})")
print(f"Mean confidence delta: {mean_delta_c:.3f}")
print(f"Sub-threshold drops:   {subthresh_c}")
print(f"\n{'='*50}")
print(f"  SENSITIVITY RATIO: {sensitivity_ratio:.2f}")
print(f"  Family A (drug):    {flip_rate_a:.1%}")
print(f"  Family C (symptom): {flip_rate_c:.1%}")
print(f"{'='*50}")
print("Saved: /content/results/family_c_with_confidence.json")

[Family C] Running symptom masking on 6821 ADE sentences...


Masking symptoms: 100%|██████████| 6821/6821 [16:24<00:00,  6.93it/s]



Sentences with symptom match: 6820
Sentences excluded (no symptom found): 1

Flip rate:             30.2% (2063/6820)
Mean confidence delta: 0.302
Sub-threshold drops:   30

  SENSITIVITY RATIO: 2.46
  Family A (drug):    74.5%
  Family C (symptom): 30.2%
Saved: /content/results/family_c_with_confidence.json


In [ ]:
from transformers import AutoModelForSequenceClassification

print("[Attention] Loading model with attention outputs...")
model_attn = AutoModelForSequenceClassification.from_pretrained(
    CONFIG['finetuned_model_dir'],
    output_attentions=True
)
model_attn = model_attn.to(DEVICE)
model_attn.eval()

# Load drug names for drug-token identification
with open('/content/extracted_drug_names.json') as f:
    drug_names = json.load(f)
drug_names_lower = set(d.lower() for d in drug_names)

# Pick 50 sentences deterministically
rng = np.random.RandomState(42)
sample_indices = rng.choice(len(all_ade_sentences), size=50, replace=False)
sample_sentences = [all_ade_sentences[i] for i in sample_indices]

def get_drug_token_mask(tokens):
    mask = []
    for tok in tokens:
        clean = tok.replace('##', '').lower().strip()
        is_drug = len(clean) > 2 and any(clean in drug for drug in drug_names_lower)
        mask.append(is_drug)
    return mask

attention_results = []

for record in tqdm(sample_sentences, desc="Attention extraction"):
    text = record['text']
    inputs = tokenizer(
        text, return_tensors='pt',
        truncation=True, max_length=128
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model_attn(**inputs)

    # Last layer, average across heads, column sum = attention received
    last_attn = outputs.attentions[-1].squeeze(0).mean(dim=0)
    attn_received = last_attn.sum(dim=0).cpu().numpy()

    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    drug_mask = get_drug_token_mask(tokens)

    drug_idx    = [i for i, d in enumerate(drug_mask) if d]
    nondrug_idx = [i for i, d in enumerate(drug_mask) if not d]

    mean_drug    = float(np.mean(attn_received[drug_idx]))    if drug_idx    else 0.0
    mean_nondrug = float(np.mean(attn_received[nondrug_idx])) if nondrug_idx else 0.0
    ratio        = mean_drug / mean_nondrug if mean_nondrug > 0 else 0.0

    attention_results.append({
        "sentence_id":        sentence_id(text),
        "text":               text,
        "tokens":             tokens,
        "drug_token_indices": drug_idx,
        "attn_received":      attn_received.tolist(),
        "mean_drug_attn":     mean_drug,
        "mean_nondrug_attn":  mean_nondrug,
        "drug_nondrug_ratio": ratio,
        "n_drug_tokens":      len(drug_idx)
    })

# Aggregate over sentences that had drug tokens
valid = [r for r in attention_results if r['n_drug_tokens'] > 0]
mean_drug_agg    = float(np.mean([r['mean_drug_attn']     for r in valid]))
mean_nondrug_agg = float(np.mean([r['mean_nondrug_attn']  for r in valid]))
mean_ratio_agg   = float(np.mean([r['drug_nondrug_ratio'] for r in valid]))

with open('/content/results/attention_summary.json', 'w') as f:
    json.dump({
        "experiment": "attention_analysis",
        "n_sentences": len(attention_results),
        "n_sentences_with_drug_tokens": len(valid),
        "aggregate": {
            "mean_drug_token_attention":    mean_drug_agg,
            "mean_nondrug_token_attention": mean_nondrug_agg,
            "mean_drug_nondrug_ratio":      mean_ratio_agg
        },
        "per_sentence": attention_results,
        "metadata": {
            "layer": "last (layer 12)",
            "aggregation": "mean across heads, column sum",
            "seed": 42,
            "timestamp": datetime.now().isoformat()
        }
    }, f, indent=2)

print(f"\nSentences processed:          {len(attention_results)}")
print(f"Sentences with drug tokens:   {len(valid)}")
print(f"Mean drug token attention:    {mean_drug_agg:.4f}")
print(f"Mean non-drug attention:      {mean_nondrug_agg:.4f}")
print(f"Drug/non-drug ratio:          {mean_ratio_agg:.2f}x")
print("Saved: /content/results/attention_summary.json")

[Attention] Loading model with attention outputs...


Attention extraction: 100%|██████████| 50/50 [00:00<00:00, 58.35it/s]


Sentences processed:          50
Sentences with drug tokens:   50
Mean drug token attention:    0.6513
Mean non-drug attention:      1.1121
Drug/non-drug ratio:          0.59x
Saved: /content/results/attention_summary.json


In [ ]:
from google.colab import files

result_files = [
    '/content/results/family_a_with_confidence.json',
    '/content/results/family_b_with_confidence.json',
    '/content/results/family_c_with_confidence.json',
    '/content/results/attention_summary.json',
    '/content/results/extracted_symptom_terms.json',
]

for fpath in result_files:
    if os.path.exists(fpath):
        print(f"Downloading {os.path.basename(fpath)}...")
        files.download(fpath)
    else:
        print(f"MISSING: {fpath}")

print("\nDone! Upload all 5 files to Claude.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Done! Upload all 5 files to Claude.


In [ ]:
!pip install statsmodels scipy matplotlib -q
import json, numpy as np, re, os, hashlib
from datetime import datetime
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.proportion import proportion_confint

os.makedirs('/content/results', exist_ok=True)
os.makedirs('/content/figures', exist_ok=True)
print("Done")

Done


In [ ]:
with open('/content/results/family_a_with_confidence.json') as f:
    fa = json.load(f)
with open('/content/results/family_b_with_confidence.json') as f:
    fb = json.load(f)
with open('/content/results/family_c_with_confidence.json') as f:
    fc = json.load(f)

# Build paired data
fa_by_id = {r['sentence_id']: r for r in fa['per_sentence']}
fc_by_id = {r['sentence_id']: r for r in fc['per_sentence']}
common_ids = list(set(fa_by_id.keys()) & set(fc_by_id.keys()))
print(f"Paired sentences: {len(common_ids)}")

# Contingency table
a_flip_c_flip = a_flip_c_noflip = a_noflip_c_flip = a_noflip_c_noflip = 0
paired = []
for sid in common_ids:
    a_f = fa_by_id[sid]['flipped']
    c_f = fc_by_id[sid]['flipped']
    paired.append((int(a_f), int(c_f)))
    if a_f and c_f:           a_flip_c_flip += 1
    elif a_f and not c_f:     a_flip_c_noflip += 1
    elif not a_f and c_f:     a_noflip_c_flip += 1
    else:                     a_noflip_c_noflip += 1

print(f"\nContingency table:")
print(f"              C-flip  C-noflip")
print(f"A-flip        {a_flip_c_flip:5d}  {a_flip_c_noflip:8d}")
print(f"A-noflip      {a_noflip_c_flip:5d}  {a_noflip_c_noflip:8d}")

table = [[a_flip_c_flip, a_flip_c_noflip],[a_noflip_c_flip, a_noflip_c_noflip]]
res = mcnemar(table, exact=False, correction=True)
print(f"\nMcNemar: statistic={res.statistic:.2f}, p={res.pvalue:.2e}, significant={res.pvalue<0.05}")

# Bootstrap CI for sensitivity ratio
rng = np.random.RandomState(42)
boot_ratios = []
n = len(paired)
for _ in range(1000):
    idx = rng.choice(n, size=n, replace=True)
    sample = [paired[i] for i in idx]
    a_r = np.mean([s[0] for s in sample])
    c_r = np.mean([s[1] for s in sample])
    if c_r > 0:
        boot_ratios.append(a_r / c_r)

ci_lo, ci_hi = np.percentile(boot_ratios, [2.5, 97.5])
print(f"\nSensitivity Ratio: {fc['sensitivity_ratio']:.2f} (95% CI: [{ci_lo:.2f}, {ci_hi:.2f}])")

# Wilson CIs
def wci(n_s, n_t):
    return proportion_confint(n_s, n_t, alpha=0.05, method='wilson')

fa_lo,fa_hi = wci(fa['n_flipped'], fa['n_sentences'])
fb_lo,fb_hi = wci(fb['n_flipped'], fb['n_sentences'])
fc_lo,fc_hi = wci(fc['n_flipped'], fc['n_sentences_with_symptom'])

print(f"\nWilson 95% CIs:")
print(f"Family A: {fa['flip_rate']:.1%}  [{fa_lo:.1%}, {fa_hi:.1%}]")
print(f"Family B: {fb['flip_rate']:.1%}  [{fb_lo:.1%}, {fb_hi:.1%}]")
print(f"Family C: {fc['flip_rate']:.1%}  [{fc_lo:.1%}, {fc_hi:.1%}]")

stats_out = {
    "mcnemar": {
        "statistic": res.statistic, "p_value": res.pvalue,
        "significant": bool(res.pvalue<0.05), "n_paired": len(common_ids),
        "contingency_table": table
    },
    "sensitivity_ratio": {
        "point_estimate": fc['sensitivity_ratio'],
        "ci_95_low": ci_lo, "ci_95_high": ci_hi
    },
    "wilson_cis": {
        "family_a": {"rate": fa['flip_rate'], "low": fa_lo, "high": fa_hi},
        "family_b": {"rate": fb['flip_rate'], "low": fb_lo, "high": fb_hi},
        "family_c": {"rate": fc['flip_rate'], "low": fc_lo, "high": fc_hi}
    }
}
with open('/content/results/statistical_tests.json', 'w') as f:
    json.dump(stats_out, f, indent=2)
print("\nSaved: /content/results/statistical_tests.json")

Paired sentences: 4270

Contingency table:
              C-flip  C-noflip
A-flip         1112      2123
A-noflip        295       740

McNemar: statistic=1380.45, p=3.72e-302, significant=True

Sensitivity Ratio: 2.46 (95% CI: [2.20, 2.41])

Wilson 95% CIs:
Family A: 74.5%  [73.5%, 75.6%]
Family B: 13.8%  [12.9%, 14.8%]
Family C: 30.2%  [29.2%, 31.4%]

Saved: /content/results/statistical_tests.json


In [ ]:
# By sentence length
def bucket_length(text):
    n = len(text.split())
    if n <= 15: return '1-15'
    elif n <= 30: return '16-30'
    elif n <= 50: return '31-50'
    else: return '51+'

length_buckets = {}
for r in fa['per_sentence']:
    b = bucket_length(r['original_text'])
    length_buckets.setdefault(b, []).append(r['flipped'])

print("=== By sentence length ===")
for b in ['1-15','16-30','31-50','51+']:
    vals = length_buckets.get(b, [])
    if vals:
        print(f"  {b}: {np.mean(vals):.1%} ({sum(vals)}/{len(vals)})")

# By drug mention count
drug_count_buckets = {}
for r in fa['per_sentence']:
    n = r['masked_text'].count('[MASK]')
    b = '1' if n==1 else '2' if n==2 else '3+'
    drug_count_buckets.setdefault(b, []).append(r['flipped'])

print("\n=== By drug mention count ===")
for b in ['1','2','3+']:
    vals = drug_count_buckets.get(b, [])
    if vals:
        print(f"  {b} drug(s): {np.mean(vals):.1%} ({sum(vals)}/{len(vals)})")

# By causal marker
causal_markers = ['induced','caused by','due to','associated with',
                  'led to','resulted in','following','after']
def has_causal(text):
    tl = text.lower()
    return any(m in tl for m in causal_markers)

causal_y = [r['flipped'] for r in fa['per_sentence'] if has_causal(r['original_text'])]
causal_n = [r['flipped'] for r in fa['per_sentence'] if not has_causal(r['original_text'])]

print("\n=== By causal marker presence ===")
print(f"  With marker:    {np.mean(causal_y):.1%} ({sum(causal_y)}/{len(causal_y)})")
print(f"  Without marker: {np.mean(causal_n):.1%} ({sum(causal_n)}/{len(causal_n)})")

# Paired failure modes
neither = [sid for sid in common_ids
           if not fa_by_id[sid]['flipped'] and not fc_by_id[sid]['flipped']]
both    = [sid for sid in common_ids
           if fa_by_id[sid]['flipped'] and fc_by_id[sid]['flipped']]
only_a  = [sid for sid in common_ids
           if fa_by_id[sid]['flipped'] and not fc_by_id[sid]['flipped']]
only_c  = [sid for sid in common_ids
           if not fa_by_id[sid]['flipped'] and fc_by_id[sid]['flipped']]

total = len(common_ids)
print(f"\n=== Paired failure modes ({total} sentences) ===")
print(f"  Both flipped:      {len(both)} ({len(both)/total:.1%})")
print(f"  Only A flipped:    {len(only_a)} ({len(only_a)/total:.1%})")
print(f"  Only C flipped:    {len(only_c)} ({len(only_c)/total:.1%})")
print(f"  Neither flipped:   {len(neither)} ({len(neither)/total:.1%})")

error_patterns = {
    "by_sentence_length": {
        b: {"flip_rate": float(np.mean(v)), "n": len(v), "flipped": sum(v)}
        for b,v in length_buckets.items()
    },
    "by_drug_mention_count": {
        b: {"flip_rate": float(np.mean(v)), "n": len(v), "flipped": sum(v)}
        for b,v in drug_count_buckets.items()
    },
    "by_causal_marker": {
        "with_marker":    {"flip_rate": float(np.mean(causal_y)), "n": len(causal_y), "flipped": sum(causal_y)},
        "without_marker": {"flip_rate": float(np.mean(causal_n)), "n": len(causal_n), "flipped": sum(causal_n)}
    },
    "paired_failure_modes": {
        "both_flipped": len(both), "only_a": len(only_a),
        "only_c": len(only_c), "neither": len(neither),
        "total_paired": total
    }
}
with open('/content/results/error_patterns.json', 'w') as f:
    json.dump(error_patterns, f, indent=2)
print("\nSaved: /content/results/error_patterns.json")

=== By sentence length ===
  1-15: 75.0% (1730/2306)
  16-30: 74.0% (2546/3440)
  31-50: 75.4% (768/1019)
  51+: 71.4% (40/56)

=== By drug mention count ===
  1 drug(s): 76.4% (3192/4177)
  2 drug(s): 71.7% (1304/1819)
  3+ drug(s): 71.3% (588/825)

=== By causal marker presence ===
  With marker:    74.5% (2649/3558)
  Without marker: 74.6% (2435/3263)

=== Paired failure modes (4270 sentences) ===
  Both flipped:      1112 (26.0%)
  Only A flipped:    2123 (49.7%)
  Only C flipped:    295 (6.9%)
  Neither flipped:   740 (17.3%)

Saved: /content/results/error_patterns.json


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

with open('/content/results/attention_summary.json') as f:
    attn = json.load(f)

TEAL  = '#2A9D8F'
CORAL = '#E76F51'
NAVY  = '#264653'
GRAY  = '#A8B4B8'
LGRAY = '#F4F4F4'

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 150
})

# ── FIG 1: Flip rates all families ──
fig, ax = plt.subplots(figsize=(10, 5))
labels = ['Family A\n(Drug Mask)\nADE', 'Family A\n(Drug Mask)\nnon-ADE',
          'Family B\n(Brand Sub)\nAll', 'Family B\n(Brand Sub)\nADE',
          'Family C\n(Symptom Mask)\nADE']
rates  = [0.745, 0.0165, 0.0889, 0.1381, 0.302]
colors = [TEAL, GRAY, CORAL, CORAL, NAVY]
ci_los = [stats_out['wilson_cis']['family_a']['low'], 0.013, 0.083, 0.129,
          stats_out['wilson_cis']['family_c']['low']]
ci_his = [stats_out['wilson_cis']['family_a']['high'], 0.020, 0.095, 0.148,
          stats_out['wilson_cis']['family_c']['high']]

x = np.arange(len(labels))
bars = ax.bar(x, [r*100 for r in rates], color=colors, width=0.55, zorder=3)
for i,(r,lo,hi) in enumerate(zip(rates,ci_los,ci_his)):
    ax.errorbar(i, r*100, yerr=[[(r-lo)*100],[(hi-r)*100]],
                fmt='none', color='#333', capsize=5, linewidth=1.5, zorder=4)
for bar,r in zip(bars,rates):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1.2,
            f'{r:.1%}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9.5)
ax.set_ylabel('Flip Rate (%)'); ax.set_ylim(0,92)
ax.set_title('Prediction Flip Rates Across All Perturbation Families',
             fontsize=13, fontweight='bold', pad=12)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'{v:.0f}%'))
ax.grid(axis='y', alpha=0.3, zorder=0); ax.set_facecolor(LGRAY)
ax.legend(handles=[
    mpatches.Patch(color=TEAL,  label='Family A – Drug Masking'),
    mpatches.Patch(color=CORAL, label='Family B – Brand Substitution'),
    mpatches.Patch(color=NAVY,  label='Family C – Symptom Masking'),
    mpatches.Patch(color=GRAY,  label='Non-ADE baseline')
], loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig('/content/figures/fig1_flip_rates.png', bbox_inches='tight')
plt.close()
print("Fig 1 saved")

# ── FIG 2: Sensitivity ratio ──
fig, ax = plt.subplots(figsize=(7, 4.5))
bars2 = ax.barh(['Family C\n(Symptom Masking)', 'Family A\n(Drug Masking)'],
                [fc['flip_rate']*100, fa['flip_rate']*100],
                color=[NAVY, TEAL], height=0.45, zorder=3)
for bar,v in zip(bars2,[fc['flip_rate']*100, fa['flip_rate']*100]):
    ax.text(v+0.8, bar.get_y()+bar.get_height()/2,
            f'{v:.1f}%', va='center', fontsize=12, fontweight='bold')
ax.text(0.97, 0.08,
        f'Sensitivity Ratio = {fc["sensitivity_ratio"]:.2f}\n'
        f'(95% CI: [{stats_out["sensitivity_ratio"]["ci_95_low"]:.2f}, '
        f'{stats_out["sensitivity_ratio"]["ci_95_high"]:.2f}])\np < 0.001',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=11,
        bbox=dict(boxstyle='round,pad=0.5', facecolor='#FFF8E7',
                  edgecolor=CORAL, linewidth=1.5))
ax.set_xlabel('Flip Rate (%)'); ax.set_xlim(0,90)
ax.set_title('Drug vs Symptom Masking: Sensitivity Ratio',
             fontsize=13, fontweight='bold', pad=12)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'{v:.0f}%'))
ax.grid(axis='x', alpha=0.3, zorder=0); ax.set_facecolor(LGRAY)
plt.tight_layout()
plt.savefig('/content/figures/fig2_sensitivity_ratio.png', bbox_inches='tight')
plt.close()
print("Fig 2 saved")

# ── FIG 3: Confidence delta distributions ──
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax2,(data,title,color) in zip(axes,[
    (fa['per_sentence'], 'Family A – Drug Masking', TEAL),
    (fb['per_sentence'], 'Family B – Brand Substitution', CORAL),
    (fc['per_sentence'], 'Family C – Symptom Masking', NAVY)
]):
    deltas = [r['confidence_delta'] for r in data]
    ax2.hist(deltas, bins=40, color=color, alpha=0.85,
             edgecolor='white', linewidth=0.4)
    ax2.axvline(np.mean(deltas), color='#333', linestyle='--',
                linewidth=1.5, label=f'Mean={np.mean(deltas):.2f}')
    ax2.axvline(0, color='red', linestyle=':', linewidth=1, alpha=0.6)
    ax2.set_title(title, fontsize=10, fontweight='bold')
    ax2.set_xlabel('Confidence Delta', fontsize=9)
    ax2.set_ylabel('Count', fontsize=9)
    ax2.legend(fontsize=8)
    ax2.set_facecolor(LGRAY)
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
fig.suptitle('Confidence Score Shifts Across Perturbation Families',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/figures/fig3_confidence_deltas.png', bbox_inches='tight')
plt.close()
print("Fig 3 saved")

# ── FIG 4: Error patterns ──
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

lengths   = ['1-15','16-30','31-50','51+']
len_rates = [error_patterns['by_sentence_length'].get(b,{}).get('flip_rate',0)*100 for b in lengths]
len_ns    = [error_patterns['by_sentence_length'].get(b,{}).get('n',0) for b in lengths]
axes[0].bar(lengths, len_rates, color=TEAL, width=0.55, zorder=3)
for i,(r,n) in enumerate(zip(len_rates,len_ns)):
    axes[0].text(i, r+0.5, f'{r:.1f}%\n(n={n})', ha='center', fontsize=8.5)
axes[0].set_title('Flip Rate by Sentence Length', fontweight='bold')
axes[0].set_ylabel('Flip Rate (%)'); axes[0].set_ylim(0,95)
axes[0].grid(axis='y',alpha=0.3,zorder=0); axes[0].set_facecolor(LGRAY)

dc_labels = ['1','2','3+']
dc_rates  = [error_patterns['by_drug_mention_count'].get(b,{}).get('flip_rate',0)*100 for b in dc_labels]
dc_ns     = [error_patterns['by_drug_mention_count'].get(b,{}).get('n',0) for b in dc_labels]
axes[1].bar(dc_labels, dc_rates, color=CORAL, width=0.55, zorder=3)
for i,(r,n) in enumerate(zip(dc_rates,dc_ns)):
    axes[1].text(i, r+0.5, f'{r:.1f}%\n(n={n})', ha='center', fontsize=8.5)
axes[1].set_title('Flip Rate by Drug Mention Count', fontweight='bold')
axes[1].set_ylabel('Flip Rate (%)'); axes[1].set_ylim(0,95)
axes[1].grid(axis='y',alpha=0.3,zorder=0); axes[1].set_facecolor(LGRAY)

fm = error_patterns['paired_failure_modes']
total_fm = fm['total_paired']
fm_labels = ['Both\nFlipped','Only Drug\nFlipped','Only Symptom\nFlipped','Neither\nFlipped']
fm_vals   = [fm['both_flipped']/total_fm*100, fm['only_a']/total_fm*100,
             fm['only_c']/total_fm*100, fm['neither']/total_fm*100]
bars4 = axes[2].bar(fm_labels, fm_vals,
                    color=[TEAL,'#4CAF75',CORAL,GRAY], width=0.55, zorder=3)
for bar,v in zip(bars4,fm_vals):
    axes[2].text(bar.get_x()+bar.get_width()/2, v+0.5,
                 f'{v:.1f}%', ha='center', fontsize=9)
axes[2].set_title('Paired Failure Mode Breakdown', fontweight='bold')
axes[2].set_ylabel('% of Paired Sentences'); axes[2].set_ylim(0,65)
axes[2].grid(axis='y',alpha=0.3,zorder=0); axes[2].set_facecolor(LGRAY)

for ax3 in axes:
    ax3.spines['top'].set_visible(False)
    ax3.spines['right'].set_visible(False)
fig.suptitle('Error Pattern Analysis — Family A (Drug Masking, ADE Sentences)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/figures/fig4_error_patterns.png', bbox_inches='tight')
plt.close()
print("Fig 4 saved")

# ── FIG 5: Attention heatmaps ──
sentences_attn = [r for r in attn['per_sentence'] if r['n_drug_tokens'] > 0]
ratios_attn = [r['drug_nondrug_ratio'] for r in sentences_attn]
sorted_idx  = np.argsort(ratios_attn)
picked = [sentences_attn[sorted_idx[-1]],
          sentences_attn[sorted_idx[len(sorted_idx)//2]],
          sentences_attn[sorted_idx[0]]]
pick_labels = ['Highest drug attention','Median drug attention','Lowest drug attention']

fig, axes5 = plt.subplots(3, 1, figsize=(12, 9))
for ax5,rec,lbl in zip(axes5,picked,pick_labels):
    tokens    = rec['tokens'][:20]
    attn_vals = np.array(rec['attn_received'][:20])
    attn_norm = attn_vals/attn_vals.max() if attn_vals.max()>0 else attn_vals
    im = ax5.imshow(attn_norm.reshape(1,-1), aspect='auto', cmap='Blues', vmin=0, vmax=1)
    ax5.set_xticks(range(len(tokens)))
    ax5.set_xticklabels(tokens, rotation=45, ha='right', fontsize=8.5)
    ax5.set_yticks([])
    ax5.set_title(f'{lbl} | ratio={rec["drug_nondrug_ratio"]:.2f}x | '
                  f'"{rec["text"][:70]}..."', fontsize=9, fontweight='bold')
    for idx in rec['drug_token_indices']:
        if idx < len(tokens):
            ax5.add_patch(plt.Rectangle((idx-0.5,-0.5),1,1,
                          fill=False,edgecolor=CORAL,linewidth=2.5))

plt.colorbar(im, ax=axes5, label='Normalized Attention Received', shrink=0.6)
fig.suptitle('BioBERT Last-Layer Attention | Red border = drug token',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/figures/fig5_attention_heatmaps.png', bbox_inches='tight')
plt.close()
print("Fig 5 saved")
print("\nAll figures saved to /content/figures/")

Fig 1 saved
Fig 2 saved
Fig 3 saved
Fig 4 saved


/tmp/ipykernel_1788/938406662.py:187: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Fig 5 saved

All figures saved to /content/figures/


In [ ]:
from google.colab import files
import os

# Copy results to Drive as backup
import shutil
drive_dst = '/content/drive/MyDrive/CS671_ADE_complete/final_results'
os.makedirs(drive_dst, exist_ok=True)
shutil.copytree('/content/results', f'{drive_dst}/results', dirs_exist_ok=True)
shutil.copytree('/content/figures', f'{drive_dst}/figures', dirs_exist_ok=True)
print("Backed up to Drive")

# Download all result files
print("\nDownloading results...")
for f in os.listdir('/content/results'):
    fpath = f'/content/results/{f}'
    print(f"  {f}")
    files.download(fpath)

# Download all figures
print("\nDownloading figures...")
for f in os.listdir('/content/figures'):
    fpath = f'/content/figures/{f}'
    print(f"  {f}")
    files.download(fpath)

print("\nAll done!")

Backed up to Drive

  error_patterns.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  family_b_with_confidence.json


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

  fig2_sensitivity_ratio.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


All done!


In [ ]:
!pip install shap -q
import shap
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# We already have model and tokenizer loaded but let's make sure
print(f"Model device: {next(model.parameters()).device}")
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")

# Pick 20 ADE-positive sentences for SHAP (it's slow, keep it small)
import random
random.seed(42)
shap_sentences = random.sample(
    [r['original_text'] for r in fa['per_sentence']],
    20
)
print(f"\nSelected {len(shap_sentences)} sentences for SHAP analysis")
print(f"Sample: {shap_sentences[0][:80]}...")

Model device: cuda:0
Tokenizer vocab size: 28996

Selected 20 sentences for SHAP analysis
Sample: Cicatricial entropion associated with chronic dipivefrin application....


In [ ]:
from transformers import pipeline
import shap, torch, json
import numpy as np
from tqdm import tqdm

# Build a prediction function SHAP can call
def predict_proba(texts):
    if isinstance(texts, np.ndarray):
        texts = texts.tolist()
    encoded = tokenizer(
        texts, padding=True, truncation=True,
        max_length=128, return_tensors='pt'
    ).to(DEVICE)
    with torch.no_grad():
        logits = model(**encoded).logits
        probs  = torch.softmax(logits, dim=-1)
    return probs.cpu().numpy()

# Build SHAP explainer using a small background set (10 sentences)
background = shap_sentences[:10]
explainer  = shap.Explainer(predict_proba, shap.maskers.Text(tokenizer))
print("Explainer built. Running SHAP on 20 sentences (takes 3-5 mins)...")

shap_values = explainer(shap_sentences, batch_size=4)
print("SHAP done!")

# Extract results
shap_results = []
for i, text in enumerate(shap_sentences):
    tokens    = shap_values.data[i]
    values    = shap_values.values[i][:, 1]  # class 1 = ADE

    # Identify which tokens are drug tokens
    tokens_lower = [t.lower().strip() for t in tokens]
    is_drug = [any(d in t or t in d for d in drug_names_lower
                   if len(t) > 2) for t in tokens_lower]

    drug_shap    = [float(v) for v,d in zip(values, is_drug) if d]
    nondrug_shap = [float(v) for v,d in zip(values, is_drug) if not d]

    mean_drug_shap    = float(np.mean(drug_shap))    if drug_shap    else 0.0
    mean_nondrug_shap = float(np.mean(nondrug_shap)) if nondrug_shap else 0.0

    # Top 3 most important tokens
    top3_idx    = np.argsort(np.abs(values))[-3:][::-1]
    top3_tokens = [(tokens[j], float(values[j])) for j in top3_idx]
    top3_are_drugs = [is_drug[j] for j in top3_idx]

    shap_results.append({
        "sentence_id":         sentence_id(text),
        "text":                text,
        "tokens":              list(tokens),
        "shap_values_ade":     [float(v) for v in values],
        "is_drug_token":       is_drug,
        "mean_drug_shap":      mean_drug_shap,
        "mean_nondrug_shap":   mean_nondrug_shap,
        "drug_nondrug_ratio":  mean_drug_shap/mean_nondrug_shap if mean_nondrug_shap!=0 else 0,
        "top3_tokens":         top3_tokens,
        "top3_are_drugs":      top3_are_drugs,
        "n_drug_tokens":       sum(is_drug)
    })

# Aggregate
valid = [r for r in shap_results if r['n_drug_tokens'] > 0]
mean_drug_shap_agg    = np.mean([r['mean_drug_shap']    for r in valid])
mean_nondrug_shap_agg = np.mean([r['mean_nondrug_shap'] for r in valid])
ratio_shap            = mean_drug_shap_agg / mean_nondrug_shap_agg if mean_nondrug_shap_agg != 0 else 0
top3_drug_pct = np.mean([any(r['top3_are_drugs']) for r in shap_results])

print(f"\n=== SHAP Results ===")
print(f"Mean drug token SHAP:     {mean_drug_shap_agg:.4f}")
print(f"Mean non-drug token SHAP: {mean_nondrug_shap_agg:.4f}")
print(f"Drug/non-drug SHAP ratio: {ratio_shap:.2f}x")
print(f"Sentences with drug in top-3: {top3_drug_pct:.1%}")

print(f"\nTop tokens per sentence:")
for r in shap_results[:5]:
    print(f"  {r['text'][:60]}...")
    for (tok, val), is_d in zip(r['top3_tokens'], r['top3_are_drugs']):
        print(f"    '{tok}' shap={val:.3f} {'[DRUG]' if is_d else ''}")

with open('/content/results/shap_analysis.json', 'w') as f:
    json.dump({
        "experiment": "shap_token_analysis",
        "n_sentences": len(shap_results),
        "aggregate": {
            "mean_drug_shap": float(mean_drug_shap_agg),
            "mean_nondrug_shap": float(mean_nondrug_shap_agg),
            "drug_nondrug_ratio": float(ratio_shap),
            "pct_sentences_drug_in_top3": float(top3_drug_pct)
        },
        "per_sentence": shap_results
    }, f, indent=2)
print("\nSaved: /content/results/shap_analysis.json")

Explainer built. Running SHAP on 20 sentences (takes 3-5 mins)...


PartitionExplainer explainer: 21it [00:40,  2.38s/it]

SHAP done!

=== SHAP Results ===
Mean drug token SHAP:     0.0377
Mean non-drug token SHAP: 0.0232
Drug/non-drug SHAP ratio: 1.62x
Sentences with drug in top-3: 75.0%

Top tokens per sentence:
  Cicatricial entropion associated with chronic dipivefrin app...
    'chronic ' shap=0.187 
    'dip' shap=0.178 [DRUG]
    'application' shap=0.177 
  Soon after introduction of insulin therapy, she developed se...
    'insulin ' shap=0.432 [DRUG]
    'ede' shap=0.071 [DRUG]
    'severe ' shap=0.062 
  The three reported cases demonstrate that troglitazone is an...
    'az' shap=0.252 
    'lit' shap=0.123 [DRUG]
    'rre' shap=0.063 
  Here we report a patient with newly diagnosed acute promyelo...
    'therapy' shap=0.142 [DRUG]
    'receiving ' shap=0.122 
    'developed ' shap=0.083 
  Treatment of APL in pregnancy is controversial as the use of...
    'use ' shap=0.219 
    'te' shap=0.129 
    'RA ' shap=0.122 

Saved: /content/results/shap_analysis.json


In [ ]:
!pip install lime -q
from lime.lime_text import LimeTextExplainer
import numpy as np
import json
from tqdm import tqdm

# Prediction function for LIME (needs numpy array output)
def lime_predict(texts):
    encoded = tokenizer(
        list(texts), padding=True, truncation=True,
        max_length=128, return_tensors='pt'
    ).to(DEVICE)
    with torch.no_grad():
        logits = model(**encoded).logits
        probs  = torch.softmax(logits, dim=-1)
    return probs.cpu().numpy()

explainer_lime = LimeTextExplainer(class_names=['non-ADE', 'ADE'])
print("LIME explainer built")

# Use same 20 sentences as SHAP
lime_results = []

for text in tqdm(shap_sentences, desc="LIME"):
    exp = explainer_lime.explain_instance(
        text,
        lime_predict,
        num_features=10,
        num_samples=500,
        labels=[1]   # explain ADE class
    )

    # Get token importances for ADE class
    token_weights = exp.as_list(label=1)  # [(token, weight), ...]

    # Check which tokens are drug-related
    top3 = token_weights[:3]
    top3_are_drugs = []
    for tok, w in top3:
        tok_lower = tok.lower().strip()
        is_d = any(tok_lower in d or d in tok_lower
                   for d in drug_names_lower if len(tok_lower) > 2)
        top3_are_drugs.append(is_d)

    # Mean weight of drug vs non-drug tokens
    drug_weights    = []
    nondrug_weights = []
    for tok, w in token_weights:
        tok_lower = tok.lower().strip()
        is_d = any(tok_lower in d or d in tok_lower
                   for d in drug_names_lower if len(tok_lower) > 2)
        if is_d:
            drug_weights.append(w)
        else:
            nondrug_weights.append(w)

    mean_drug_lime    = float(np.mean(drug_weights))    if drug_weights    else 0.0
    mean_nondrug_lime = float(np.mean(nondrug_weights)) if nondrug_weights else 0.0

    lime_results.append({
        "sentence_id":        sentence_id(text),
        "text":               text,
        "token_weights":      [(t, float(w)) for t,w in token_weights],
        "top3_tokens":        top3,
        "top3_are_drugs":     top3_are_drugs,
        "mean_drug_lime":     mean_drug_lime,
        "mean_nondrug_lime":  mean_nondrug_lime,
        "drug_nondrug_ratio": mean_drug_lime/mean_nondrug_lime if mean_nondrug_lime != 0 else 0,
        "n_drug_features":    len(drug_weights)
    })

# Aggregate
valid_lime     = [r for r in lime_results if r['n_drug_features'] > 0]
mean_drug_agg  = float(np.mean([r['mean_drug_lime']    for r in valid_lime]))
mean_ndrug_agg = float(np.mean([r['mean_nondrug_lime'] for r in valid_lime]))
ratio_lime     = mean_drug_agg / mean_ndrug_agg if mean_ndrug_agg != 0 else 0
top3_drug_pct  = float(np.mean([any(r['top3_are_drugs']) for r in lime_results]))

print(f"\n=== LIME Results ===")
print(f"Mean drug token weight:     {mean_drug_agg:.4f}")
print(f"Mean non-drug token weight: {mean_ndrug_agg:.4f}")
print(f"Drug/non-drug LIME ratio:   {ratio_lime:.2f}x")
print(f"Sentences with drug in top-3: {top3_drug_pct:.1%}")

print(f"\nTop tokens per sentence (first 5):")
for r in lime_results[:5]:
    print(f"  {r['text'][:60]}...")
    for (tok,w), is_d in zip(r['top3_tokens'], r['top3_are_drugs']):
        print(f"    '{tok}' weight={w:.3f} {'[DRUG]' if is_d else ''}")

with open('/content/results/lime_analysis.json', 'w') as f:
    json.dump({
        "experiment": "lime_token_analysis",
        "n_sentences": len(lime_results),
        "aggregate": {
            "mean_drug_lime":     mean_drug_agg,
            "mean_nondrug_lime":  mean_ndrug_agg,
            "drug_nondrug_ratio": ratio_lime,
            "pct_sentences_drug_in_top3": top3_drug_pct
        },
        "per_sentence": lime_results
    }, f, indent=2)
print("\nSaved: /content/results/lime_analysis.json")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 11.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
LIME explainer built


LIME: 100%|██████████| 20/20 [00:02<00:00,  9.80it/s]


=== LIME Results ===
Mean drug token weight:     0.2939
Mean non-drug token weight: 0.0734
Drug/non-drug LIME ratio:   4.00x
Sentences with drug in top-3: 90.0%

Top tokens per sentence (first 5):
  Cicatricial entropion associated with chronic dipivefrin app...
    'dipivefrin' weight=0.727 [DRUG]
    'entropion' weight=0.257 [DRUG]
    'Cicatricial' weight=0.234 
  Soon after introduction of insulin therapy, she developed se...
    'insulin' weight=0.931 [DRUG]
    'developed' weight=0.062 
    'after' weight=0.053 
  The three reported cases demonstrate that troglitazone is an...
    'troglitazone' weight=0.906 [DRUG]
    'irreversible' weight=0.092 
    'lead' weight=0.087 [DRUG]
  Here we report a patient with newly diagnosed acute promyelo...
    'after' weight=0.338 
    'acid' weight=0.161 [DRUG]
    'developed' weight=0.137 
  Treatment of APL in pregnancy is controversial as the use of...
    'teratogenic' weight=0.592 [DRUG]
    'ATRA' weight=0.472 [DRUG]
    'of' weight=0.

In [ ]:
!pip install captum -q
import captum
print(f"Captum version: {captum.__version__}")
print("Ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 25.0 MB/s eta 0:00:00
Captum version: 0.9.0
Ready


In [ ]:
from captum.attr import LayerIntegratedGradients
import torch
import numpy as np
import json
from tqdm import tqdm

# Wrapper so captum can access embeddings
def predict_for_captum(input_ids, attention_mask, token_type_ids):
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        token_type_ids=token_type_ids
    )
    return torch.softmax(outputs.logits, dim=-1)

# Integrated gradients on the embedding layer
lig = LayerIntegratedGradients(
    predict_for_captum,
    model.bert.embeddings
)

def get_ig_attributions(text):
    encoded = tokenizer(
        text, return_tensors='pt',
        truncation=True, max_length=128,
        padding='max_length'
    )
    input_ids      = encoded['input_ids'].to(DEVICE)
    attention_mask = encoded['attention_mask'].to(DEVICE)
    token_type_ids = encoded['token_type_ids'].to(DEVICE)

    # Baseline = all [PAD] tokens
    baseline_ids = torch.zeros_like(input_ids).to(DEVICE)

    attributions, delta = lig.attribute(
        inputs=(input_ids, attention_mask, token_type_ids),
        baselines=(baseline_ids, attention_mask, token_type_ids),
        target=1,  # ADE class
        additional_forward_args=None,
        return_convergence_delta=True,
        n_steps=50,
        internal_batch_size=4
    )

    # Sum across embedding dim, take absolute value
    attr_sum = attributions.squeeze(0).sum(dim=-1).detach().cpu().numpy()
    tokens   = tokenizer.convert_ids_to_tokens(input_ids[0])

    # Remove padding
    real_len = int(attention_mask.sum().item())
    tokens   = tokens[:real_len]
    attr_sum = attr_sum[:real_len]

    return tokens, attr_sum, float(delta.item())

ig_results = []

for text in tqdm(shap_sentences, desc="Integrated Gradients"):
    try:
        tokens, attrs, delta = get_ig_attributions(text)
        tokens_lower = [t.replace('##','').lower().strip() for t in tokens]
        is_drug = [any(t in d or d in t for d in drug_names_lower
                       if len(t) > 2) for t in tokens_lower]

        drug_attrs    = [float(a) for a,d in zip(attrs, is_drug) if d]
        nondrug_attrs = [float(a) for a,d in zip(attrs, is_drug) if not d]

        mean_drug_ig    = float(np.mean(drug_attrs))    if drug_attrs    else 0.0
        mean_nondrug_ig = float(np.mean(nondrug_attrs)) if nondrug_attrs else 0.0

        # Top 3 tokens by attribution magnitude
        top3_idx    = np.argsort(np.abs(attrs))[-3:][::-1]
        top3_tokens = [(tokens[j], float(attrs[j])) for j in top3_idx]
        top3_are_drugs = [is_drug[j] for j in top3_idx]

        ig_results.append({
            "sentence_id":        sentence_id(text),
            "text":               text,
            "tokens":             tokens,
            "attributions":       [float(a) for a in attrs],
            "is_drug_token":      is_drug,
            "mean_drug_ig":       mean_drug_ig,
            "mean_nondrug_ig":    mean_nondrug_ig,
            "drug_nondrug_ratio": mean_drug_ig/mean_nondrug_ig if mean_nondrug_ig!=0 else 0,
            "top3_tokens":        top3_tokens,
            "top3_are_drugs":     top3_are_drugs,
            "convergence_delta":  delta,
            "n_drug_tokens":      sum(is_drug)
        })
    except Exception as e:
        print(f"Skipped: {text[:50]}... Error: {e}")

# Aggregate
valid_ig       = [r for r in ig_results if r['n_drug_tokens'] > 0]
mean_drug_agg  = float(np.mean([r['mean_drug_ig']      for r in valid_ig]))
mean_ndrug_agg = float(np.mean([r['mean_nondrug_ig']   for r in valid_ig]))
ratio_ig       = mean_drug_agg / mean_ndrug_agg if mean_ndrug_agg != 0 else 0
top3_drug_pct  = float(np.mean([any(r['top3_are_drugs']) for r in ig_results]))

print(f"\n=== Integrated Gradients Results ===")
print(f"Mean drug token attribution:     {mean_drug_agg:.4f}")
print(f"Mean non-drug token attribution: {mean_ndrug_agg:.4f}")
print(f"Drug/non-drug IG ratio:          {ratio_ig:.2f}x")
print(f"Sentences with drug in top-3:    {top3_drug_pct:.1%}")

print(f"\nTop tokens per sentence (first 5):")
for r in ig_results[:5]:
    print(f"  {r['text'][:60]}...")
    for (tok,w), is_d in zip(r['top3_tokens'], r['top3_are_drugs']):
        print(f"    '{tok}' attr={w:.3f} {'[DRUG]' if is_d else ''}")

with open('/content/results/integrated_gradients.json', 'w') as f:
    json.dump({
        "experiment": "integrated_gradients",
        "n_sentences": len(ig_results),
        "aggregate": {
            "mean_drug_ig":       mean_drug_agg,
            "mean_nondrug_ig":    mean_ndrug_agg,
            "drug_nondrug_ratio": ratio_ig,
            "pct_sentences_drug_in_top3": top3_drug_pct
        },
        "per_sentence": ig_results
    }, f, indent=2)
print("\nSaved: /content/results/integrated_gradients.json")

Integrated Gradients: 100%|██████████| 20/20 [00:08<00:00,  2.36it/s]


=== Integrated Gradients Results ===
Mean drug token attribution:     0.0416
Mean non-drug token attribution: 0.0256
Drug/non-drug IG ratio:          1.63x
Sentences with drug in top-3:    95.0%

Top tokens per sentence (first 5):
  Cicatricial entropion associated with chronic dipivefrin app...
    '##rin' attr=0.000 [DRUG]
    '##f' attr=0.000 
    'dip' attr=0.000 [DRUG]
  Soon after introduction of insulin therapy, she developed se...
    'insulin' attr=2.602 [DRUG]
    'developed' attr=1.369 
    'after' attr=1.355 
  The three reported cases demonstrate that troglitazone is an...
    '##one' attr=0.139 [DRUG]
    '##az' attr=0.077 
    'lead' attr=0.037 [DRUG]
  Here we report a patient with newly diagnosed acute promyelo...
    'acid' attr=0.000 [DRUG]
    'trans' attr=0.000 [DRUG]
    '##tino' attr=0.000 [DRUG]
  Treatment of APL in pregnancy is controversial as the use of...
    '##ra' attr=0.177 
    '##genic' attr=0.101 
    '##rato' attr=0.094 [DRUG]

Saved: /content/resul

In [ ]:
import re, json, numpy as np
from tqdm import tqdm

# Load drug names
with open('/content/extracted_drug_names.json') as f:
    drug_names_list = json.load(f)
drug_names_sorted = sorted(drug_names_list, key=len, reverse=True)

def find_drug_spans(text, drug_names_sorted):
    """Find all drug matches in text, return list of (drug, start, end)."""
    found = []
    for drug in drug_names_sorted:
        pattern = re.compile(r'\b' + re.escape(drug) + r'\b', re.IGNORECASE)
        for m in pattern.finditer(text):
            # Avoid overlapping matches
            if not any(s <= m.start() < e for _,s,e in found):
                found.append((drug, m.start(), m.end()))
    return found

def mask_specific_drugs(text, drugs_to_mask, drug_names_sorted):
    """Mask only a specific subset of drug spans."""
    masked = text
    for drug in sorted(drugs_to_mask, key=len, reverse=True):
        pattern = re.compile(r'\b' + re.escape(drug) + r'\b', re.IGNORECASE)
        masked = pattern.sub('[MASK]', masked)
    return masked

# Use flip sentences from Family A (those that flipped)
# Take 200 sentences that flipped for minimality analysis
import random
random.seed(42)
flip_records = [r for r in fa['per_sentence'] if r['flipped']]
sample_records = random.sample(flip_records, min(200, len(flip_records)))
print(f"Running minimality on {len(sample_records)} flip sentences...")

minimality_results = []

for record in tqdm(sample_records, desc="Minimality"):
    text = record['original_text']
    spans = find_drug_spans(text, drug_names_sorted)
    drug_matches = [drug for drug,_,_ in spans]

    if not drug_matches:
        continue

    # Check if masking each drug individually flips prediction
    single_flip = {}
    for drug in drug_matches:
        masked = mask_specific_drugs(text, [drug], drug_names_sorted)
        _, probs = predict_batch([masked])
        pred = int(probs[0] >= 0.5)
        single_flip[drug] = (pred == 0)  # flipped to non-ADE

    # Minimum number of drugs needed to flip
    min_needed = None
    if any(single_flip.values()):
        min_needed = 1
    elif len(drug_matches) > 1:
        # Try pairs
        from itertools import combinations
        flipped_pair = False
        for combo in combinations(drug_matches, 2):
            masked = mask_specific_drugs(text, list(combo), drug_names_sorted)
            _, probs = predict_batch([masked])
            pred = int(probs[0] >= 0.5)
            if pred == 0:
                flipped_pair = True
                break
        min_needed = 2 if flipped_pair else len(drug_matches)
    else:
        min_needed = len(drug_matches)

    minimality_results.append({
        "sentence_id":      sentence_id(text),
        "text":             text,
        "n_drugs":          len(drug_matches),
        "drug_matches":     drug_matches,
        "single_drug_flip": single_flip,
        "min_drugs_to_flip": min_needed,
        "any_single_flip":  any(single_flip.values())
    })

# Aggregate
total = len(minimality_results)
single_sufficient = sum(1 for r in minimality_results if r['any_single_flip'])
multi_drug = [r for r in minimality_results if r['n_drugs'] > 1]
redundant  = sum(1 for r in multi_drug if not r['any_single_flip'])

min_counts = {}
for r in minimality_results:
    k = str(r['min_drugs_to_flip'])
    min_counts[k] = min_counts.get(k, 0) + 1

print(f"\n=== Minimality Results ===")
print(f"Total sentences analyzed:        {total}")
print(f"Single drug sufficient to flip:  {single_sufficient} ({single_sufficient/total:.1%})")
print(f"Multi-drug sentences:            {len(multi_drug)}")
print(f"Redundant (multi needed):        {redundant} ({redundant/max(len(multi_drug),1):.1%})")
print(f"\nMin drugs needed distribution:")
for k in sorted(min_counts.keys()):
    print(f"  {k} drug(s): {min_counts[k]} ({min_counts[k]/total:.1%})")

with open('/content/results/minimality_analysis.json', 'w') as f:
    json.dump({
        "experiment": "counterfactual_minimality",
        "n_sentences": total,
        "aggregate": {
            "pct_single_drug_sufficient": single_sufficient/total,
            "pct_redundant_multi_drug": redundant/max(len(multi_drug),1),
            "min_drug_distribution": min_counts
        },
        "per_sentence": minimality_results
    }, f, indent=2)
print("\nSaved: /content/results/minimality_analysis.json")

Running minimality on 200 flip sentences...


Minimality: 100%|██████████| 200/200 [00:13<00:00, 14.68it/s]


=== Minimality Results ===
Total sentences analyzed:        200
Single drug sufficient to flip:  164 (82.0%)
Multi-drug sentences:            73
Redundant (multi needed):        36 (49.3%)

Min drugs needed distribution:
  1 drug(s): 164 (82.0%)
  2 drug(s): 28 (14.0%)
  3 drug(s): 6 (3.0%)
  4 drug(s): 1 (0.5%)
  5 drug(s): 1 (0.5%)

Saved: /content/results/minimality_analysis.json


In [ ]:
import re, json, numpy as np
from tqdm import tqdm

# Negation insertion patterns
# For each pattern, we insert a negation word after the trigger
NEGATION_RULES = [
    (r'\b(developed)\b',       'did not develop'),
    (r'\b(presented with)\b',  'did not present with'),
    (r'\b(showed)\b',          'did not show'),
    (r'\b(experienced)\b',     'did not experience'),
    (r'\b(reported)\b',        'did not report'),
    (r'\b(caused)\b',          'did not cause'),
    (r'\b(induced)\b',         'did not induce'),
    (r'\b(resulted in)\b',     'did not result in'),
    (r'\b(associated with)\b', 'not associated with'),
    (r'\b(following)\b',       'not following'),
]

def insert_negation(text):
    """
    Try each negation rule. Return (negated_text, rule_used)
    for the first matching rule, or (None, None) if no rule matches.
    """
    for pattern, replacement in NEGATION_RULES:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            negated = text[:match.start()] + replacement + text[match.end():]
            return negated, pattern
    return None, None

# Use 300 ADE-positive sentences
import random
random.seed(42)
sample = random.sample(fa['per_sentence'], min(300, len(fa['per_sentence'])))

negation_results = []
excluded_no_match = 0

for record in tqdm(sample, desc="Negation"):
    text = record['original_text']
    negated_text, rule = insert_negation(text)

    if negated_text is None:
        excluded_no_match += 1
        continue

    # Run inference on original and negated
    _, orig_probs  = predict_batch([text])
    _, negat_probs = predict_batch([negated_text])

    orig_pred  = int(orig_probs[0]  >= 0.5)
    negat_pred = int(negat_probs[0] >= 0.5)
    flipped    = (orig_pred != negat_pred)

    negation_results.append({
        "sentence_id":      sentence_id(text),
        "original_text":    text,
        "negated_text":     negated_text,
        "rule_used":        rule,
        "original_pred":    orig_pred,
        "negated_pred":     negat_pred,
        "original_prob":    float(orig_probs[0]),
        "negated_prob":     float(negat_probs[0]),
        "confidence_delta": float(orig_probs[0] - negat_probs[0]),
        "flipped":          flipped
    })

# Aggregate
total     = len(negation_results)
n_flipped = sum(1 for r in negation_results if r['flipped'])
flip_rate = n_flipped / total
mean_delta = float(np.mean([r['confidence_delta'] for r in negation_results]))
subthresh  = sum(1 for r in negation_results
                 if r['confidence_delta'] >= 0.10 and not r['flipped'])

# By rule
rule_stats = {}
for r in negation_results:
    rule = r['rule_used']
    rule_stats.setdefault(rule, []).append(r['flipped'])

print(f"\n=== Negation Counterfactual Results ===")
print(f"Sentences with negation match:  {total}")
print(f"Excluded (no pattern matched):  {excluded_no_match}")
print(f"Flip rate after negation:       {flip_rate:.1%} ({n_flipped}/{total})")
print(f"Mean confidence delta:          {mean_delta:.3f}")
print(f"Sub-threshold drops (≥0.10):    {subthresh}")

print(f"\nBy negation rule:")
for rule, flips in sorted(rule_stats.items(), key=lambda x:-np.mean(x[1])):
    print(f"  {rule:35s} flip={np.mean(flips):.1%} (n={len(flips)})")

print(f"\nExamples that DID NOT flip (model ignores negation):")
no_flip = [r for r in negation_results if not r['flipped']][:3]
for r in no_flip:
    print(f"  Original: {r['original_text'][:80]}")
    print(f"  Negated:  {r['negated_text'][:80]}")
    print(f"  Prob: {r['original_prob']:.2f} → {r['negated_prob']:.2f}")
    print()

print(f"Examples that DID flip:")
did_flip = [r for r in negation_results if r['flipped']][:3]
for r in did_flip:
    print(f"  Original: {r['original_text'][:80]}")
    print(f"  Negated:  {r['negated_text'][:80]}")
    print(f"  Prob: {r['original_prob']:.2f} → {r['negated_prob']:.2f}")
    print()

with open('/content/results/negation_analysis.json', 'w') as f:
    json.dump({
        "experiment": "negation_counterfactuals",
        "n_sentences": total,
        "n_excluded_no_match": excluded_no_match,
        "n_flipped": n_flipped,
        "flip_rate": flip_rate,
        "mean_confidence_delta": mean_delta,
        "subthreshold_drops_gte_010": subthresh,
        "by_rule": {
            rule: {"flip_rate": float(np.mean(flips)), "n": len(flips)}
            for rule, flips in rule_stats.items()
        },
        "per_sentence": negation_results
    }, f, indent=2)
print("Saved: /content/results/negation_analysis.json")

Negation: 100%|██████████| 300/300 [00:04<00:00, 61.87it/s]


=== Negation Counterfactual Results ===
Sentences with negation match:  152
Excluded (no pattern matched):  148
Flip rate after negation:       57.2% (87/152)
Mean confidence delta:          0.569
Sub-threshold drops (≥0.10):    2

By negation rule:
  \b(presented with)\b                flip=100.0% (n=3)
  \b(showed)\b                        flip=100.0% (n=1)
  \b(developed)\b                     flip=74.5% (n=55)
  \b(experienced)\b                   flip=66.7% (n=3)
  \b(induced)\b                       flip=59.1% (n=44)
  \b(caused)\b                        flip=57.1% (n=7)
  \b(resulted in)\b                   flip=50.0% (n=2)
  \b(associated with)\b               flip=38.9% (n=18)
  \b(following)\b                     flip=20.0% (n=5)
  \b(reported)\b                      flip=7.1% (n=14)

Examples that DID NOT flip (model ignores negation):
  Original: Cicatricial entropion associated with chronic dipivefrin application.
  Negated:  Cicatricial entropion not associated with chro

In [ ]:
import shap, torch, json
import numpy as np
from tqdm import tqdm

# Use smaller set — interactions are expensive
import random
random.seed(42)
interaction_sentences = random.sample(shap_sentences, 10)
print(f"Running SHAP interactions on {len(interaction_sentences)} sentences...")

def predict_proba_np(texts):
    if isinstance(texts, np.ndarray):
        texts = texts.tolist()
    encoded = tokenizer(
        texts, padding=True, truncation=True,
        max_length=128, return_tensors='pt'
    ).to(DEVICE)
    with torch.no_grad():
        logits = model(**encoded).logits
        probs  = torch.softmax(logits, dim=-1)
    return probs.cpu().numpy()

explainer_int = shap.Explainer(
    predict_proba_np,
    shap.maskers.Text(tokenizer)
)

interaction_results = []

for text in tqdm(interaction_sentences, desc="SHAP interactions"):
    sv = explainer_int([text], batch_size=4)
    tokens    = list(sv.data[0])
    values    = sv.values[0][:, 1]  # ADE class

    tokens_lower = [t.lower().strip() for t in tokens]
    is_drug = [any(t in d or d in t for d in drug_names_lower
                   if len(t) > 2) for t in tokens_lower]

    drug_idx    = [i for i,d in enumerate(is_drug) if d]
    nondrug_idx = [i for i,d in enumerate(is_drug) if not d]

    # Top drug token and top symptom token by SHAP magnitude
    top_drug_tok    = None
    top_nondrug_tok = None
    top_drug_val    = 0.0
    top_nondrug_val = 0.0

    for i in drug_idx:
        if abs(values[i]) > abs(top_drug_val):
            top_drug_val = float(values[i])
            top_drug_tok = tokens[i]

    for i in nondrug_idx:
        if abs(values[i]) > abs(top_nondrug_val):
            top_nondrug_val = float(values[i])
            top_nondrug_tok = tokens[i]

    # Interaction proxy: do drug and non-drug tokens push in same direction?
    drug_vals    = [float(values[i]) for i in drug_idx]
    nondrug_vals = [float(values[i]) for i in nondrug_idx]

    same_direction = 0
    total_pairs    = 0
    for dv in drug_vals:
        for ndv in nondrug_vals:
            total_pairs += 1
            if (dv > 0 and ndv > 0) or (dv < 0 and ndv < 0):
                same_direction += 1

    interaction_rate = same_direction / total_pairs if total_pairs > 0 else 0.0

    interaction_results.append({
        "sentence_id":       sentence_id(text),
        "text":              text,
        "tokens":            tokens,
        "shap_values":       [float(v) for v in values],
        "is_drug_token":     is_drug,
        "top_drug_token":    top_drug_tok,
        "top_drug_shap":     top_drug_val,
        "top_nondrug_token": top_nondrug_tok,
        "top_nondrug_shap":  top_nondrug_val,
        "interaction_rate":  interaction_rate,
        "n_drug_tokens":     len(drug_idx),
        "n_nondrug_tokens":  len(nondrug_idx)
    })
    print(f"  '{text[:50]}...'")
    print(f"    Top drug:    '{top_drug_tok}' ({top_drug_val:+.3f})")
    print(f"    Top context: '{top_nondrug_tok}' ({top_nondrug_val:+.3f})")
    print(f"    Same-direction pairs: {interaction_rate:.1%}")

# Aggregate
mean_interaction = float(np.mean([r['interaction_rate'] for r in interaction_results]))
mean_drug_shap   = float(np.mean([r['top_drug_shap']    for r in interaction_results
                                   if r['top_drug_token']]))
mean_ctx_shap    = float(np.mean([abs(r['top_nondrug_shap']) for r in interaction_results
                                   if r['top_nondrug_token']]))

print(f"\n=== SHAP Interaction Results ===")
print(f"Mean same-direction rate:     {mean_interaction:.1%}")
print(f"Mean top drug SHAP:           {mean_drug_shap:+.3f}")
print(f"Mean top context SHAP (abs):  {mean_ctx_shap:.3f}")
print(f"\nInterpretation:")
if mean_interaction > 0.6:
    print("  Drug and context tokens push predictions in the SAME direction")
    print("  => Model uses both jointly (cooperative)")
else:
    print("  Drug and context tokens often push in OPPOSITE directions")
    print("  => Model shows tension between drug signal and context signal")

with open('/content/results/shap_interactions.json', 'w') as f:
    json.dump({
        "experiment": "shap_interaction_values",
        "n_sentences": len(interaction_results),
        "aggregate": {
            "mean_same_direction_rate": mean_interaction,
            "mean_top_drug_shap":       mean_drug_shap,
            "mean_top_context_shap":    mean_ctx_shap
        },
        "per_sentence": interaction_results
    }, f, indent=2)
print("\nSaved: /content/results/shap_interactions.json")

Running SHAP interactions on 10 sentences...


SHAP interactions:  10%|█         | 1/10 [00:02<00:19,  2.15s/it]

  'Here we report a patient with newly diagnosed acut...'
    Top drug:    'therapy' (+0.142)
    Top context: 'receiving ' (+0.122)
    Same-direction pairs: 81.2%


SHAP interactions:  20%|██        | 2/10 [00:04<00:17,  2.17s/it]

  'Cicatricial entropion associated with chronic dipi...'
    Top drug:    'dip' (+0.178)
    Top context: 'chronic ' (+0.187)
    Same-direction pairs: 70.1%


...

  'Soon after introduction of insulin therapy, she de...'
    Top drug:    'insulin ' (+0.432)
    Top context: 'severe ' (+0.062)
    Same-direction pairs: 81.2%


SHAP interactions:  90%|█████████ | 9/10 [00:18<00:02,  2.02s/it]

  'PATIENT AND METHOD: A 34-year-old woman with chron...'
    Top drug:    'AND ' (+0.052)
    Top context: 'developed ' (+0.094)
    Same-direction pairs: 87.1%


SHAP interactions: 100%|██████████| 10/10 [00:20<00:00,  2.05s/it]

  'The polycystic changes disappeared from the ovarie...'
    Top drug:    'and ' (+0.047)
    Top context: 'p' (+0.051)
    Same-direction pairs: 95.7%

=== SHAP Interaction Results ===
Mean same-direction rate:     88.6%
Mean top drug SHAP:           +0.129
Mean top context SHAP (abs):  0.135

Interpretation:
  Drug and context tokens push predictions in the SAME direction
  => Model uses both jointly (cooperative)

Saved: /content/results/shap_interactions.json


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

TEAL  = '#2A9D8F'
CORAL = '#E76F51'
NAVY  = '#264653'
GRAY  = '#A8B4B8'
LGRAY = '#F4F4F4'

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 150
})

# ── FIG 6: Method comparison ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Drug/non-drug ratio across all methods
methods = ['Attention\n(Last Layer)', 'SHAP', 'Integrated\nGradients',
           'Counterfactual\n(Flip Rate)', 'LIME']
ratios  = [0.59, 1.62, 1.63, 2.46, 4.00]
colors  = [GRAY, TEAL, TEAL, NAVY, CORAL]

bars = axes[0].bar(methods, ratios, color=colors, width=0.55, zorder=3)
axes[0].axhline(1.0, color='red', linestyle='--', linewidth=1.5,
                alpha=0.7, label='Ratio = 1.0 (no preference)')
axes[0].axhline(2.0, color=NAVY, linestyle=':', linewidth=1.5,
                alpha=0.5, label='Success threshold (2.0)')
for bar, v in zip(bars, ratios):
    axes[0].text(bar.get_x()+bar.get_width()/2,
                 v+0.05, f'{v:.2f}×',
                 ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_ylabel('Drug / Non-drug Token Importance Ratio')
axes[0].set_title('Drug Token Importance\nAcross Explanation Methods',
                  fontweight='bold', fontsize=12)
axes[0].set_ylim(0, 5.2)
axes[0].legend(fontsize=8, loc='upper left')
axes[0].grid(axis='y', alpha=0.3, zorder=0)
axes[0].set_facecolor(LGRAY)

# Right: Drug in top-3 across methods + negation + minimality
metrics  = ['SHAP\nDrug in Top-3', 'LIME\nDrug in Top-3',
            'IG\nDrug in Top-3', 'Negation\nFlip Rate',
            'Minimality\n1-drug sufficient']
values   = [75.0, 90.0, 95.0, 57.2, 82.0]
colors2  = [TEAL, CORAL, TEAL, NAVY, GRAY]

bars2 = axes[1].bar(metrics, values, color=colors2, width=0.55, zorder=3)
for bar, v in zip(bars2, values):
    axes[1].text(bar.get_x()+bar.get_width()/2,
                 v+0.8, f'{v:.1f}%',
                 ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[1].set_ylabel('Percentage (%)')
axes[1].set_title('Key Metrics Across\nAll XAI Experiments',
                  fontweight='bold', fontsize=12)
axes[1].set_ylim(0, 112)
axes[1].yaxis.set_major_formatter(
    plt.FuncFormatter(lambda v,_: f'{v:.0f}%'))
axes[1].grid(axis='y', alpha=0.3, zorder=0)
axes[1].set_facecolor(LGRAY)

fig.suptitle('Comprehensive XAI Analysis — BioBERT ADE Detection',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/figures/fig6_method_comparison.png', bbox_inches='tight')
plt.close()
print("Fig 6 saved")

# ── FIG 7: Negation analysis by rule ──
fig, ax = plt.subplots(figsize=(10, 5))

with open('/content/results/negation_analysis.json') as f:
    neg = json.load(f)

rules      = list(neg['by_rule'].keys())
rule_rates = [neg['by_rule'][r]['flip_rate']*100 for r in rules]
rule_ns    = [neg['by_rule'][r]['n'] for r in rules]

# Clean up rule labels
clean_labels = [r.replace(r'\b','').replace('(','').replace(')','').strip()
                for r in rules]
sorted_pairs = sorted(zip(rule_rates, clean_labels, rule_ns), reverse=True)
rule_rates, clean_labels, rule_ns = zip(*sorted_pairs)

bar_colors = [CORAL if r < 50 else TEAL for r in rule_rates]
bars3 = ax.barh(clean_labels, rule_rates, color=bar_colors,
                height=0.55, zorder=3)
ax.axvline(50, color='red', linestyle='--', linewidth=1.5,
           alpha=0.6, label='50% threshold')
for bar, v, n in zip(bars3, rule_rates, rule_ns):
    ax.text(v+0.5, bar.get_y()+bar.get_height()/2,
            f'{v:.1f}% (n={n})', va='center', fontsize=9)
ax.set_xlabel('Flip Rate After Negation (%)')
ax.set_title('Negation Counterfactuals — Flip Rate by Linguistic Pattern\n'
             'Low flip rate = model ignores that negation',
             fontweight='bold', fontsize=12)
ax.set_xlim(0, 125)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'{v:.0f}%'))
ax.legend(fontsize=9)
ax.grid(axis='x', alpha=0.3, zorder=0)
ax.set_facecolor(LGRAY)
plt.tight_layout()
plt.savefig('/content/figures/fig7_negation_by_rule.png', bbox_inches='tight')
plt.close()
print("Fig 7 saved")

# ── FIG 8: Minimality distribution ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

with open('/content/results/minimality_analysis.json') as f:
    mini = json.load(f)

dist   = mini['aggregate']['min_drug_distribution']
labels = sorted(dist.keys())
vals   = [dist[k] for k in labels]
total  = sum(vals)
pcts   = [v/total*100 for v in vals]

bars4 = axes[0].bar(labels, pcts, color=TEAL, width=0.55, zorder=3)
for bar, v, p in zip(bars4, vals, pcts):
    axes[0].text(bar.get_x()+bar.get_width()/2,
                 p+0.5, f'{p:.1f}%\n(n={v})',
                 ha='center', va='bottom', fontsize=9)
axes[0].set_xlabel('Minimum Drug Tokens Needed to Flip')
axes[0].set_ylabel('% of Sentences')
axes[0].set_title('Counterfactual Minimality\nDistribution',
                  fontweight='bold')
axes[0].set_ylim(0, 100)
axes[0].grid(axis='y', alpha=0.3, zorder=0)
axes[0].set_facecolor(LGRAY)

# SHAP interaction pie
interaction_rate = 88.6
axes[1].pie(
    [interaction_rate, 100-interaction_rate],
    labels=[f'Same direction\n(cooperative)\n{interaction_rate:.1f}%',
            f'Opposite direction\n(competitive)\n{100-interaction_rate:.1f}%'],
    colors=[TEAL, CORAL],
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
axes[1].set_title('SHAP Token Interactions\nDrug vs Context Tokens',
                  fontweight='bold')

fig.suptitle('Minimality & Interaction Analysis',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/figures/fig8_minimality_interactions.png',
            bbox_inches='tight')
plt.close()
print("Fig 8 saved")
print("\nAll figures done!")

# ── Save to Drive + Download ──
import shutil, os
from google.colab import files

drive_dst = '/content/drive/MyDrive/CS671_ADE_complete/final_results'
os.makedirs(drive_dst, exist_ok=True)
shutil.copytree('/content/results', f'{drive_dst}/results', dirs_exist_ok=True)
shutil.copytree('/content/figures', f'{drive_dst}/figures', dirs_exist_ok=True)
print("\nBacked up to Drive")

print("\nDownloading new results...")
new_results = [
    'shap_analysis.json', 'lime_analysis.json',
    'integrated_gradients.json', 'minimality_analysis.json',
    'negation_analysis.json', 'shap_interactions.json'
]
for fname in new_results:
    fpath = f'/content/results/{fname}'
    if os.path.exists(fpath):
        print(f"  {fname}")
        files.download(fpath)

print("\nDownloading new figures...")
new_figures = [
    'fig6_method_comparison.png',
    'fig7_negation_by_rule.png',
    'fig8_minimality_interactions.png'
]
for fname in new_figures:
    fpath = f'/content/figures/{fname}'
    if os.path.exists(fpath):
        print(f"  {fname}")
        files.download(fpath)

print("\nAll done!")

Fig 6 saved
Fig 7 saved
Fig 8 saved

All figures done!

Backed up to Drive

  shap_analysis.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  lime_analysis.json


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

  fig8_minimality_interactions.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


All done!


In [25]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

os.makedirs('/content/results', exist_ok=True)
os.makedirs('/content/output', exist_ok=True)

src = '/content/drive/MyDrive/CS671_ADE_complete'

items = [
    'trained_model',
    'extracted_drug_names.json',
    'full_drug_brand_mapping.json',
    'classification_data_with_brands.parquet',
    'masking_flip_examples_ade_only.json',
    'masking_non_flip_examples_ade_only.json',
]

for item in items:
    s = f'{src}/{item}'
    d = f'/content/{item}'
    if os.path.exists(s):
        if os.path.isdir(s):
            shutil.copytree(s, d, dirs_exist_ok=True)
        else:
            shutil.copy2(s, d)
        print(f"✓ {item}")
    else:
        print(f"✗ MISSING: {item}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ trained_model
✓ extracted_drug_names.json
✓ full_drug_brand_mapping.json
✓ classification_data_with_brands.parquet
✓ masking_flip_examples_ade_only.json
✓ masking_non_flip_examples_ade_only.json


In [26]:
import hashlib

# Load model
tokenizer, model = load_tokenizer_and_model(CONFIG['finetuned_model_dir'], num_labels=2)
model = model.to(DEVICE)
model.eval()
model_wrapper = create_model_wrapper(model, tokenizer)

# Batched inference function
def predict_batch(texts, batch_size=64):
    all_labels, all_probs = [], []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(batch, padding=True, truncation=True,
                           max_length=128, return_tensors='pt')
        encoded = {k: v.to(DEVICE) for k, v in encoded.items()}
        with torch.no_grad():
            outputs = model(**encoded)
            probs = torch.softmax(outputs.logits, dim=-1)
            labels = torch.argmax(probs, dim=-1)
        all_labels.extend(labels.cpu().numpy().tolist())
        all_probs.extend(probs[:, 1].cpu().numpy().tolist())
    return np.array(all_labels), np.array(all_probs)

def sentence_id(text):
    return hashlib.sha1(text.encode()).hexdigest()[:12]

# Load ADE sentences
with open('/content/masking_flip_examples_ade_only.json') as f:
    family_a_flips = json.load(f)
with open('/content/masking_non_flip_examples_ade_only.json') as f:
    family_a_nonflips = json.load(f)
all_ade_sentences = family_a_flips + family_a_nonflips

# Load drug names
with open('/content/extracted_drug_names.json') as f:
    drug_names_list = json.load(f)
drug_names_lower = set(d.lower() for d in drug_names_list)

# Verify
print(f"✓ Model on: {next(model.parameters()).device}")
print(f"✓ ADE sentences: {len(all_ade_sentences)}")
print(f"✓ Drug names: {len(drug_names_lower)}")

loading tokenizer from ./trained_model
loading model from ./trained_model
loaded successfully
Model is wrapped successfully.
✓ Model on: cuda:0
✓ ADE sentences: 6821
✓ Drug names: 1049


In [28]:
!pip install captum -q
print("captum installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 18.4 MB/s eta 0:00:00
captum installed


In [32]:
from captum.attr import LayerIntegratedGradients
from datetime import datetime

print("[IG] Starting on all 6,821 sentences...")

def predict_for_captum(input_ids, attention_mask, token_type_ids):
    return torch.softmax(model(input_ids=input_ids,
        attention_mask=attention_mask,
        token_type_ids=token_type_ids).logits, dim=-1)

lig = LayerIntegratedGradients(predict_for_captum, model.bert.embeddings)

ig_results_full = []
ig_errors = 0

for record in tqdm(all_ade_sentences, desc="IG"):
    text = record['text']
    try:
        encoded = tokenizer(text, return_tensors='pt', truncation=True,
                           max_length=128, padding='max_length')
        input_ids      = encoded['input_ids'].to(DEVICE)
        attention_mask = encoded['attention_mask'].to(DEVICE)
        token_type_ids = encoded['token_type_ids'].to(DEVICE)
        baseline_ids   = torch.zeros_like(input_ids).to(DEVICE)

        attributions, delta = lig.attribute(
            inputs=(input_ids, attention_mask, token_type_ids),
            baselines=(baseline_ids, attention_mask, token_type_ids),
            target=1, return_convergence_delta=True,
            n_steps=50, internal_batch_size=8
        )
        attr_sum = attributions.squeeze(0).sum(dim=-1).detach().cpu().numpy()
        tokens   = tokenizer.convert_ids_to_tokens(input_ids[0])
        real_len = int(attention_mask.sum().item())
        tokens   = tokens[:real_len]
        attr_sum = attr_sum[:real_len]

        tokens_lower = [t.replace('##','').lower().strip() for t in tokens]
        is_drug = [any(t in d or d in t for d in drug_names_lower
                       if len(t) > 2) for t in tokens_lower]

        drug_attrs    = [float(a) for a,d in zip(attr_sum, is_drug) if d]
        nondrug_attrs = [float(a) for a,d in zip(attr_sum, is_drug) if not d]
        mean_drug    = float(np.mean(drug_attrs))    if drug_attrs    else 0.0
        mean_nondrug = float(np.mean(nondrug_attrs)) if nondrug_attrs else 0.0

        top3_idx       = np.argsort(np.abs(attr_sum))[-3:][::-1]
        top3_tokens    = [(tokens[j], float(attr_sum[j])) for j in top3_idx]
        top3_are_drugs = [is_drug[j] for j in top3_idx]

        ig_results_full.append({
            "sentence_id":        sentence_id(text),
            "index":              record['index'],
            "mean_drug_ig":       mean_drug,
            "mean_nondrug_ig":    mean_nondrug,
            "drug_nondrug_ratio": mean_drug / mean_nondrug if mean_nondrug != 0 else 0.0,
            "top3_tokens":        top3_tokens,
            "top3_are_drugs":     top3_are_drugs,
            "n_drug_tokens":      sum(is_drug)
        })
    except Exception as e:
        ig_errors += 1

# Aggregate
valid_ig     = [r for r in ig_results_full if r['n_drug_tokens'] > 0]
mean_drug_ag = float(np.mean([r['mean_drug_ig']    for r in valid_ig]))
mean_nd_ag   = float(np.mean([r['mean_nondrug_ig'] for r in valid_ig]))
ratio_ig     = mean_drug_ag / mean_nd_ag if mean_nd_ag != 0 else 0
top3_pct     = float(np.mean([any(r['top3_are_drugs']) for r in ig_results_full]))

print(f"\n=== IG Full Results ===")
print(f"Sentences processed:   {len(ig_results_full)}")
print(f"Errors skipped:        {ig_errors}")
print(f"Drug/non-drug ratio:   {ratio_ig:.2f}x")
print(f"Drug in top-3:         {top3_pct:.1%}")

os.makedirs('/content/results', exist_ok=True)
with open('/content/results/integrated_gradients_full.json', 'w') as f:
    json.dump({
        "experiment": "integrated_gradients_full",
        "n_sentences": len(ig_results_full),
        "n_errors": ig_errors,
        "aggregate": {
            "mean_drug_ig":       mean_drug_ag,
            "mean_nondrug_ig":    mean_nd_ag,
            "drug_nondrug_ratio": ratio_ig,
            "pct_drug_in_top3":   top3_pct
        },
        "per_sentence": ig_results_full,
        "metadata": {"n_steps": 50, "seed": 42,
                     "timestamp": datetime.now().isoformat()}
    }, f, indent=2)
print("Saved: /content/results/integrated_gradients_full.json")

[IG] Starting on all 6,821 sentences...


IG:   0%|          | 0/6821 [00:00<?, ?it/s]


=== IG Full Results ===
Sentences processed:   6821
Errors skipped:        0
Drug/non-drug ratio:   1.54x
Drug in top-3:         80.3%
Saved: /content/results/integrated_gradients_full.json


In [33]:
!pip install shap -q

import shap

def predict_proba(texts):
    if isinstance(texts, np.ndarray):
        texts = texts.tolist()
    encoded = tokenizer(texts, padding=True, truncation=True,
                       max_length=128, return_tensors='pt')
    encoded = {k: v.to(DEVICE) for k, v in encoded.items()}
    with torch.no_grad():
        logits = model(**encoded).logits
        probs  = torch.softmax(logits, dim=-1)
    return probs.cpu().numpy()

explainer_shap = shap.Explainer(predict_proba, shap.maskers.Text(tokenizer))
print("SHAP explainer ready. Running on all 6,821 sentences (~45-60 min on A100)...")

shap_results_full = []
shap_errors = 0

for record in tqdm(all_ade_sentences, desc="SHAP"):
    text = record['text']
    try:
        sv = explainer_shap([text], batch_size=8)
        tokens = list(sv.data[0])
        values = sv.values[0][:, 1]  # ADE class

        tokens_lower = [t.lower().strip() for t in tokens]
        is_drug = [any(t in d or d in t for d in drug_names_lower
                       if len(t) > 2) for t in tokens_lower]

        drug_vals    = [float(v) for v,d in zip(values, is_drug) if d]
        nondrug_vals = [float(v) for v,d in zip(values, is_drug) if not d]
        mean_drug    = float(np.mean(drug_vals))    if drug_vals    else 0.0
        mean_nondrug = float(np.mean(nondrug_vals)) if nondrug_vals else 0.0

        top3_idx       = np.argsort(np.abs(values))[-3:][::-1]
        top3_tokens    = [(tokens[j], float(values[j])) for j in top3_idx]
        top3_are_drugs = [is_drug[j] for j in top3_idx]

        shap_results_full.append({
            "sentence_id":        sentence_id(text),
            "index":              record['index'],
            "mean_drug_shap":     mean_drug,
            "mean_nondrug_shap":  mean_nondrug,
            "drug_nondrug_ratio": mean_drug / mean_nondrug if mean_nondrug != 0 else 0.0,
            "top3_tokens":        top3_tokens,
            "top3_are_drugs":     top3_are_drugs,
            "n_drug_tokens":      sum(is_drug)
        })
    except Exception as e:
        shap_errors += 1

# Aggregate
valid_shap   = [r for r in shap_results_full if r['n_drug_tokens'] > 0]
mean_drug_ag = float(np.mean([r['mean_drug_shap']    for r in valid_shap]))
mean_nd_ag   = float(np.mean([r['mean_nondrug_shap'] for r in valid_shap]))
ratio_shap   = mean_drug_ag / mean_nd_ag if mean_nd_ag != 0 else 0
top3_pct     = float(np.mean([any(r['top3_are_drugs']) for r in shap_results_full]))

print(f"\n=== SHAP Full Results ===")
print(f"Sentences processed:   {len(shap_results_full)}")
print(f"Errors skipped:        {shap_errors}")
print(f"Drug/non-drug ratio:   {ratio_shap:.2f}x")
print(f"Drug in top-3:         {top3_pct:.1%}")

with open('/content/results/shap_analysis_full.json', 'w') as f:
    json.dump({
        "experiment": "shap_full",
        "n_sentences": len(shap_results_full),
        "n_errors": shap_errors,
        "aggregate": {
            "mean_drug_shap":     mean_drug_ag,
            "mean_nondrug_shap":  mean_nd_ag,
            "drug_nondrug_ratio": ratio_shap,
            "pct_drug_in_top3":   top3_pct
        },
        "per_sentence": shap_results_full,
        "metadata": {"seed": 42, "timestamp": datetime.now().isoformat()}
    }, f, indent=2)
print("Saved: /content/results/shap_analysis_full.json")

SHAP explainer ready. Running on all 6,821 sentences (~45-60 min on A100)...


SHAP:   0%|          | 0/6821 [00:00<?, ?it/s]


=== SHAP Full Results ===
Sentences processed:   6821
Errors skipped:        0
Drug/non-drug ratio:   1.23x
Drug in top-3:         67.9%
Saved: /content/results/shap_analysis_full.json


In [34]:
!pip install lime -q
from lime.lime_text import LimeTextExplainer
from datetime import datetime

explainer_lime = LimeTextExplainer(class_names=['non-ADE', 'ADE'])

def lime_predict(texts):
    encoded = tokenizer(list(texts), padding=True, truncation=True,
                       max_length=128, return_tensors='pt')
    encoded = {k: v.to(DEVICE) for k, v in encoded.items()}
    with torch.no_grad():
        logits = model(**encoded).logits
        probs  = torch.softmax(logits, dim=-1)
    return probs.cpu().numpy()

print("LIME running on all 6,821 sentences (~30-40 min on A100)...")

lime_results_full = []
lime_errors = 0

for record in tqdm(all_ade_sentences, desc="LIME"):
    text = record['text']
    try:
        exp = explainer_lime.explain_instance(
            text, lime_predict,
            num_features=10, num_samples=500, labels=[1]
        )
        token_weights = exp.as_list(label=1)

        drug_weights    = []
        nondrug_weights = []
        for tok, w in token_weights:
            tok_lower = tok.lower().strip()
            is_d = any(tok_lower in d or d in tok_lower
                       for d in drug_names_lower if len(tok_lower) > 2)
            if is_d:
                drug_weights.append(w)
            else:
                nondrug_weights.append(w)

        mean_drug    = float(np.mean(drug_weights))    if drug_weights    else 0.0
        mean_nondrug = float(np.mean(nondrug_weights)) if nondrug_weights else 0.0

        top3 = token_weights[:3]
        top3_are_drugs = []
        for tok, w in top3:
            tok_lower = tok.lower().strip()
            is_d = any(tok_lower in d or d in tok_lower
                       for d in drug_names_lower if len(tok_lower) > 2)
            top3_are_drugs.append(is_d)

        lime_results_full.append({
            "sentence_id":        sentence_id(text),
            "index":              record['index'],
            "mean_drug_lime":     mean_drug,
            "mean_nondrug_lime":  mean_nondrug,
            "drug_nondrug_ratio": mean_drug / mean_nondrug if mean_nondrug != 0 else 0.0,
            "top3_tokens":        top3,
            "top3_are_drugs":     top3_are_drugs,
            "n_drug_features":    len(drug_weights)
        })
    except Exception as e:
        lime_errors += 1

# Aggregate
valid_lime   = [r for r in lime_results_full if r['n_drug_features'] > 0]
mean_drug_ag = float(np.mean([r['mean_drug_lime']    for r in valid_lime]))
mean_nd_ag   = float(np.mean([r['mean_nondrug_lime'] for r in valid_lime]))
ratio_lime   = mean_drug_ag / mean_nd_ag if mean_nd_ag != 0 else 0
top3_pct     = float(np.mean([any(r['top3_are_drugs']) for r in lime_results_full]))

print(f"\n=== LIME Full Results ===")
print(f"Sentences processed:   {len(lime_results_full)}")
print(f"Errors skipped:        {lime_errors}")
print(f"Drug/non-drug ratio:   {ratio_lime:.2f}x")
print(f"Drug in top-3:         {top3_pct:.1%}")

with open('/content/results/lime_analysis_full.json', 'w') as f:
    json.dump({
        "experiment": "lime_full",
        "n_sentences": len(lime_results_full),
        "n_errors": lime_errors,
        "aggregate": {
            "mean_drug_lime":     mean_drug_ag,
            "mean_nondrug_lime":  mean_nd_ag,
            "drug_nondrug_ratio": ratio_lime,
            "pct_drug_in_top3":   top3_pct
        },
        "per_sentence": lime_results_full,
        "metadata": {"seed": 42, "timestamp": datetime.now().isoformat()}
    }, f, indent=2)
print("Saved: /content/results/lime_analysis_full.json")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 12.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
LIME running on all 6,821 sentences (~30-40 min on A100)...


LIME:   0%|          | 0/6821 [00:00<?, ?it/s]


=== LIME Full Results ===
Sentences processed:   6821
Errors skipped:        0
Drug/non-drug ratio:   2.87x
Drug in top-3:         95.3%
Saved: /content/results/lime_analysis_full.json


In [35]:
from google.colab import files

for f in ['integrated_gradients_full.json',
          'shap_analysis_full.json',
          'lime_analysis_full.json']:
    files.download(f'/content/results/{f}')
    print(f"Downloaded: {f}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: integrated_gradients_full.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: shap_analysis_full.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: lime_analysis_full.json
